In [21]:
import json
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

# -----------------------------
# paths
# -----------------------------
RESULTS_PATH = Path("/home/akapociu/ift/interactiondynamics/results/ds_sweep_results_6ep_3seed.jsonl")
SUMMARY_PATH = Path("/home/akapociu/ift/interactiondynamics/results/ds_sweep_summary_6ep_3seed.jsonl")

EXPECTED_DATASETS = ["Wikipedia", "Reddit", "MOOC", "LastFM"]
EXPECTED_AGGS = ["ift", "hopfield", "settransformer", "sum", "deepsets"]
EXPECTED_UPDATES = ["ift_update", "tgn_gru", "lnn", "hopfield_update", "hnn"]
EXPECTED_SEEDS = [0, 42, 123]
EXPECTED_EPOCHS = 6

# one basic option for each HP in this sanity sweep
BASE_NON_IFT = "do=0.0|sdo=0.0|time=False"
IFT_EXTRA = "softplus|dt=0.05|gamma=0.0|k0=1.0|cap=none"


# -----------------------------
# utilities
# -----------------------------
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def nested_get(d, keys, default=np.nan):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def maybe_cast(v):
    """Cast strings like '0.0', 'True', '42' into Python values when sensible."""
    if isinstance(v, (int, float, bool)) or v is None:
        return v
    if not isinstance(v, str):
        return v

    if v == "True":
        return True
    if v == "False":
        return False
    if v == "none":
        return "none"

    try:
        if "." in v or "e" in v.lower():
            return float(v)
        return int(v)
    except Exception:
        return v

def split_run_name(full_name: str):
    """
    Example:
      Wikipedia__agg=ift|update=ift_update|do=0.0|sdo=0.0|time=False|softplus|dt=0.05|gamma=0.0|k0=1.0|cap=none
    """
    if "__" in full_name:
        dataset, config = full_name.split("__", 1)
    else:
        dataset, config = np.nan, full_name
    return dataset, config

def parse_config_string(config: str) -> dict:
    """
    Parse the config substring after dataset__ into columns.
    Handles both key=value parts and the unlabeled IFT token like 'softplus'.
    """
    out = {}
    parts = config.split("|")

    unlabeled = []
    for part in parts:
        if "=" in part:
            k, v = part.split("=", 1)
            out[k] = maybe_cast(v)
        else:
            unlabeled.append(part)

    # current sweep only has one unlabeled token we care about: ift_kappa_param
    if len(unlabeled) >= 1:
        # usually softplus or exp
        out["ift_kappa_param"] = unlabeled[0]

    # nice renamed columns
    if "agg" in out:
        out["aggregator"] = out["agg"]
    if "do" in out:
        out["dropout"] = out["do"]
    if "sdo" in out:
        out["scorer_dropout"] = out["sdo"]
    if "time" in out:
        out["use_time_features"] = out["time"]
    if "k0" in out:
        out["ift_kappa_init"] = out["k0"]

    # interpret cap
    # cap=none means cap disabled
    if "cap" in out:
        if out["cap"] == "none":
            out["ift_kappa_cap"] = False
            out["ift_kappa_max"] = np.nan
        else:
            out["ift_kappa_cap"] = True
            out["ift_kappa_max"] = maybe_cast(out["cap"])

    return out

def add_run_parsing(df: pd.DataFrame, run_col: str) -> pd.DataFrame:
    df = df.copy()
    parsed = df[run_col].apply(split_run_name)
    df["dataset_from_name"] = parsed.apply(lambda x: x[0])
    df["config_name"] = parsed.apply(lambda x: x[1])

    parsed_cfg = df["config_name"].apply(parse_config_string).apply(pd.Series)
    df = pd.concat([df, parsed_cfg], axis=1)

    # canonical columns
    if "dataset" not in df.columns:
        df["dataset"] = df["dataset_from_name"]
    else:
        df["dataset"] = df["dataset"].fillna(df["dataset_from_name"])

    return df

def make_expected_config_names():
    names = []
    for agg, upd in product(EXPECTED_AGGS, EXPECTED_UPDATES):
        base = f"agg={agg}|update={upd}|{BASE_NON_IFT}"
        if upd == "ift_update":
            base = f"{base}|{IFT_EXTRA}"
        names.append(base)
    return names

def make_expected_runs():
    cfgs = make_expected_config_names()
    rows = []
    for dataset, seed, cfg in product(EXPECTED_DATASETS, EXPECTED_SEEDS, cfgs):
        rows.append({
            "dataset": dataset,
            "seed": seed,
            "config_name": cfg,
            "run_name_full": f"{dataset}__{cfg}",
        })
    return pd.DataFrame(rows)


# -----------------------------
# load raw files
# -----------------------------
results_raw = read_jsonl(RESULTS_PATH)
summary_raw = read_jsonl(SUMMARY_PATH)

results = add_run_parsing(results_raw, run_col="run")
summary = add_run_parsing(summary_raw, run_col="name")

print("results rows:", len(results))
print("summary rows:", len(summary))

display(results.head(2))
display(summary.head(2))

results rows: 1786
summary rows: 295


,dataset,run,seed,model_cfg,train_cfg_overrides,epoch,train_step,train_eval,val,test,...,k0,cap,ift_kappa_param,aggregator,dropout,scorer_dropout,use_time_features,ift_kappa_init,ift_kappa_cap,ift_kappa_max
0,Wikipedia,Wikipedia__agg=ift|update=ift_update|do=0.0|sd...,0,"{'node_dim': 128, 'msg_dim': 128, 'event_dim':...",{},1,"{'loss': 1.4859164863652576, 'mrr': 0.81226166...","{'loss': 1.4077840197063365, 'mrr': 0.80889412...","{'loss': 1.5812949391928586, 'mrr': 0.81433618...","{'loss': 1.587320971169642, 'mrr': 0.828179197...",...,1.0,none,softplus,ift,0.0,0.0,False,1.0,False,NaN
1,Wikipedia,Wikipedia__agg=ift|update=ift_update|do=0.0|sd...,0,"{'node_dim': 128, 'msg_dim': 128, 'event_dim':...",{},2,"{'loss': 1.3546772245497143, 'mrr': 0.80758680...","{'loss': 1.5271981615781325, 'mrr': 0.80280572...","{'loss': 1.6377066335894845, 'mrr': 0.81608129...","{'loss': 1.6287850657744067, 'mrr': 0.82975872...",...,1.0,none,softplus,ift,0.0,0.0,False,1.0,False,NaN


,dataset,name,seed,epochs,best_val_mrr,best_epoch,best_snapshot,final_snapshot,wall_sec,dataset_from_name,...,k0,cap,ift_kappa_param,aggregator,dropout,scorer_dropout,use_time_features,ift_kappa_init,ift_kappa_cap,ift_kappa_max
0,Wikipedia,Wikipedia__agg=ift|update=ift_update|do=0.0|sd...,0,6,0.816081,2,"{'epoch': 2, 'train_step': {'loss': 1.35467722...","{'epoch': 6, 'train_step': {'loss': 1.34040566...",47.741141,Wikipedia,...,1.0,none,softplus,ift,0.0,0.0,False,1.0,False,NaN
1,Wikipedia,Wikipedia__agg=ift|update=ift_update|do=0.0|sd...,42,6,0.816991,1,"{'epoch': 1, 'train_step': {'loss': 1.50197236...","{'epoch': 6, 'train_step': {'loss': 1.33059256...",47.286345,Wikipedia,...,1.0,none,softplus,ift,0.0,0.0,False,1.0,False,NaN


In [22]:
# -----------------------------
# expected runs
# -----------------------------
expected_runs = make_expected_runs()

print("Expected completed runs in summary:", len(expected_runs))  # 300
print("Expected epoch rows in results if all complete:", len(expected_runs) * EXPECTED_EPOCHS)  # 1800

# -----------------------------
# summary presence
# -----------------------------
summary_keys = (
    summary[["dataset", "seed", "config_name"]]
    .drop_duplicates()
    .assign(in_summary=True)
)

# -----------------------------
# results epoch counts
# -----------------------------
results_counts = (
    results.groupby(["dataset", "seed", "config_name"], dropna=False)
    .agg(
        epoch_rows=("epoch", "size"),
        min_epoch=("epoch", "min"),
        max_epoch=("epoch", "max"),
        unique_epochs=("epoch", lambda s: tuple(sorted(pd.unique(s))))
    )
    .reset_index()
)

# -----------------------------
# merge into one audit table
# -----------------------------
audit = (
    expected_runs
    .merge(summary_keys, on=["dataset", "seed", "config_name"], how="left")
    .merge(results_counts, on=["dataset", "seed", "config_name"], how="left")
)

audit["in_summary"] = audit["in_summary"].fillna(False)
audit["epoch_rows"] = audit["epoch_rows"].fillna(0).astype(int)

def classify_results_rows(n):
    if n == 0:
        return "no_result_rows"
    elif n < EXPECTED_EPOCHS:
        return "partial_result_rows"
    elif n == EXPECTED_EPOCHS:
        return "full_result_rows"
    else:
        return "unexpected_extra_rows"

audit["results_status"] = audit["epoch_rows"].apply(classify_results_rows)

def overall_status(row):
    if row["in_summary"] and row["epoch_rows"] == EXPECTED_EPOCHS:
        return "complete"
    elif (not row["in_summary"]) and (0 < row["epoch_rows"] < EXPECTED_EPOCHS):
        return "partial_failure_or_interrupt"
    elif (not row["in_summary"]) and (row["epoch_rows"] == 0):
        return "missing_everywhere"
    elif row["in_summary"] and row["epoch_rows"] != EXPECTED_EPOCHS:
        return "weird_summary_results_mismatch"
    else:
        return "other"

audit["overall_status"] = audit.apply(overall_status, axis=1)

# -----------------------------
# high-level audit summaries
# -----------------------------
print("\n=== Overall audit counts ===")
display(audit["overall_status"].value_counts(dropna=False).rename_axis("overall_status").reset_index(name="count"))

print("\n=== Summary presence ===")
display(audit["in_summary"].value_counts(dropna=False).rename_axis("in_summary").reset_index(name="count"))

print("\n=== Results row status ===")
display(audit["results_status"].value_counts(dropna=False).rename_axis("results_status").reset_index(name="count"))

print("\n=== Audit by dataset ===")
audit_by_dataset = (
    audit.groupby("dataset", dropna=False)
    .agg(
        expected_runs=("run_name_full", "size"),
        completed=("overall_status", lambda s: (s == "complete").sum()),
        partial_failures=("overall_status", lambda s: (s == "partial_failure_or_interrupt").sum()),
        missing_everywhere=("overall_status", lambda s: (s == "missing_everywhere").sum()),
        weird_mismatch=("overall_status", lambda s: (s == "weird_summary_results_mismatch").sum()),
    )
    .reset_index()
)
display(audit_by_dataset)

print("\n=== Missing from summary ===")
missing_from_summary = audit.loc[~audit["in_summary"]].copy()
display(missing_from_summary[["dataset", "seed", "config_name", "epoch_rows", "results_status", "overall_status"]]
        .sort_values(["dataset", "config_name", "seed"])
        .head(50))

print("\n=== Partial runs in results (fewer than 6 epoch rows) ===")
partial_results = audit.loc[audit["results_status"] == "partial_result_rows"].copy()
display(partial_results[["dataset", "seed", "config_name", "epoch_rows", "min_epoch", "max_epoch", "unique_epochs", "overall_status"]]
        .sort_values(["dataset", "config_name", "seed"])
        .head(50))

Expected completed runs in summary: 300
Expected epoch rows in results if all complete: 1800

=== Overall audit counts ===


/tmp/ipykernel_2322922/3905344545.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  audit["in_summary"] = audit["in_summary"].fillna(False)


,overall_status,count
0,complete,293
1,missing_everywhere,3
2,partial_failure_or_interrupt,2
3,weird_summary_results_mismatch,2



=== Summary presence ===


,in_summary,count
0,True,295
1,False,5


...

,dataset,expected_runs,completed,partial_failures,missing_everywhere,weird_mismatch
0,LastFM,75,70,0,3,2
1,MOOC,75,75,0,0,0
2,Reddit,75,73,2,0,0
3,Wikipedia,75,75,0,0,0



=== Missing from summary ===


,dataset,seed,config_name,epoch_rows,results_status,overall_status
230,LastFM,0,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,0,no_result_rows,missing_everywhere
255,LastFM,42,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,0,no_result_rows,missing_everywhere
280,LastFM,123,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,0,no_result_rows,missing_everywhere
105,Reddit,42,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,3,partial_result_rows,partial_failure_or_interrupt
130,Reddit,123,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,3,partial_result_rows,partial_failure_or_interrupt



=== Partial runs in results (fewer than 6 epoch rows) ===


,dataset,seed,config_name,epoch_rows,min_epoch,max_epoch,unique_epochs,overall_status
105,Reddit,42,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,3,1.0,3.0,"(1, 2, 3)",partial_failure_or_interrupt
130,Reddit,123,agg=hopfield|update=ift_update|do=0.0|sdo=0.0|...,3,1.0,3.0,"(1, 2, 3)",partial_failure_or_interrupt


In [23]:
# -----------------------------
# extract run-level metrics from summary
# -----------------------------
run_df = summary.copy()

run_df["best_test_mrr"] = run_df["best_snapshot"].apply(lambda d: nested_get(d, ["test", "mrr"]))
run_df["final_test_mrr"] = run_df["final_snapshot"].apply(lambda d: nested_get(d, ["test", "mrr"]))
run_df["val_at_end"] = run_df["final_snapshot"].apply(lambda d: nested_get(d, ["val", "mrr"]))

run_df["best_val_loss"] = run_df["best_snapshot"].apply(lambda d: nested_get(d, ["val", "loss"]))
run_df["final_val_loss"] = run_df["final_snapshot"].apply(lambda d: nested_get(d, ["val", "loss"]))

run_df["best_train_eval_mrr"] = run_df["best_snapshot"].apply(lambda d: nested_get(d, ["train_eval", "mrr"]))
run_df["final_train_eval_mrr"] = run_df["final_snapshot"].apply(lambda d: nested_get(d, ["train_eval", "mrr"]))

run_df["overfit_gap"] = run_df["best_test_mrr"] - run_df["final_test_mrr"]
run_df["val_drop"] = run_df["best_val_mrr"] - run_df["val_at_end"]
run_df["peaked_early"] = run_df["best_epoch"] <= 2

# by definition, if it made it into summary, it finished the run
run_df["finished_all_epochs"] = True

# optional: merge in results epoch counts just for extra sanity
run_df = run_df.merge(
    results_counts[["dataset", "seed", "config_name", "epoch_rows"]],
    on=["dataset", "seed", "config_name"],
    how="left"
)

# nice column order
preferred_cols = [
    "dataset",
    "seed",
    "aggregator",
    "update",
    "dropout",
    "scorer_dropout",
    "use_time_features",
    "ift_kappa_param",
    "dt",
    "gamma",
    "ift_kappa_init",
    "ift_kappa_cap",
    "ift_kappa_max",
    "best_val_mrr",
    "best_epoch",
    "best_test_mrr",
    "final_test_mrr",
    "val_at_end",
    "overfit_gap",
    "val_drop",
    "peaked_early",
    "wall_sec",
    "epoch_rows",
    "finished_all_epochs",
    "config_name",
]
existing_cols = [c for c in preferred_cols if c in run_df.columns]
run_df = run_df[existing_cols + [c for c in run_df.columns if c not in existing_cols]]

print("Run-level table shape:", run_df.shape)
display(run_df.head(10))

Run-level table shape: (295, 40)


,dataset,seed,aggregator,update,dropout,scorer_dropout,use_time_features,ift_kappa_param,dt,gamma,...,agg,do,sdo,time,k0,cap,best_val_loss,final_val_loss,best_train_eval_mrr,final_train_eval_mrr
0,Wikipedia,0,ift,ift_update,0.0,0.0,False,softplus,0.05,0.0,...,ift,0.0,0.0,False,1.0,none,1.637707,1.922606,0.802806,0.768351
1,Wikipedia,42,ift,ift_update,0.0,0.0,False,softplus,0.05,0.0,...,ift,0.0,0.0,False,1.0,none,1.575965,2.024122,0.810063,0.768520
2,Wikipedia,123,ift,ift_update,0.0,0.0,False,softplus,0.05,0.0,...,ift,0.0,0.0,False,1.0,none,1.564592,1.958902,0.810775,0.767847
3,Wikipedia,0,ift,tgn_gru,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,0.995157,1.007637,0.865966,0.864671
4,Wikipedia,42,ift,tgn_gru,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,0.880190,0.965295,0.868595,0.865643
5,Wikipedia,123,ift,tgn_gru,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,1.623302,1.623302,0.864381,0.864381
6,Wikipedia,0,ift,lnn,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,2.956084,2.833595,0.719712,0.639237
7,Wikipedia,42,ift,lnn,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,2.816417,2.802197,0.720070,0.701377
8,Wikipedia,123,ift,lnn,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,3.017786,2.844499,0.738072,0.598256
9,Wikipedia,0,ift,hopfield_update,0.0,0.0,False,NaN,NaN,NaN,...,ift,0.0,0.0,False,NaN,NaN,0.947082,0.970504,0.865930,0.864897


In [24]:
# -----------------------------
# leaderboard across seeds
# -----------------------------
leaderboard = (
    run_df.groupby(["dataset", "aggregator", "update"], dropna=False)
    .agg(
        n_completed=("seed", "size"),
        seed_list=("seed", lambda s: tuple(sorted(s))),
        mean_best_val_mrr=("best_val_mrr", "mean"),
        std_best_val_mrr=("best_val_mrr", "std"),
        mean_best_test_mrr=("best_test_mrr", "mean"),
        std_best_test_mrr=("best_test_mrr", "std"),
        mean_final_test_mrr=("final_test_mrr", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_wall_sec=("wall_sec", "mean"),
        mean_overfit_gap=("overfit_gap", "mean"),
        mean_val_drop=("val_drop", "mean"),
        frac_peaked_early=("peaked_early", "mean"),
    )
    .reset_index()
)

# optional: add completion rate relative to expected 3 seeds
leaderboard["completion_rate"] = leaderboard["n_completed"] / len(EXPECTED_SEEDS)

# sort within each dataset by validation first
leaderboards_by_dataset = {
    ds: (
        leaderboard[leaderboard["dataset"] == ds]
        .sort_values(["mean_best_val_mrr", "mean_best_test_mrr"], ascending=False)
        .reset_index(drop=True)
    )
    for ds in EXPECTED_DATASETS
}

# Example:
display(leaderboards_by_dataset["Reddit"])

print("Leaderboard shape:", leaderboard.shape)
display(leaderboard.head(30))

,dataset,aggregator,update,n_completed,seed_list,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,mean_final_test_mrr,mean_best_epoch,mean_wall_sec,mean_overfit_gap,mean_val_drop,frac_peaked_early,completion_rate
0,Reddit,deepsets,ift_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,2.000000,54.240344,0.000000,0.000000,1.000000,1.000000
1,Reddit,ift,tgn_gru,3,"(0, 42, 123)",0.939561,0.003510,0.940129,0.003214,0.937349,5.333333,50.313569,0.002779,0.003074,0.000000,1.000000
2,Reddit,ift,hopfield_update,3,"(0, 42, 123)",0.924985,0.005281,0.926777,0.003281,0.897946,4.333333,544.409674,0.028831,0.036423,0.000000,1.000000
3,Reddit,ift,hnn,3,"(0, 42, 123)",0.920888,0.006690,0.923994,0.005208,0.923501,3.000000,69.798048,0.000493,0.000365,0.666667,1.000000
4,Reddit,ift,lnn,3,"(0, 42, 123)",0.915876,0.001166,0.919047,0.001441,0.919047,6.000000,60.554952,0.000000,0.000000,0.000000,1.000000
5,Reddit,deepsets,tgn_gru,3,"(0, 42, 123)",0.913801,0.005084,0.912834,0.004072,0.910328,5.000000,48.917371,0.002507,0.002273,0.000000,1.000000
6,Reddit,sum,hopfield_update,3,"(0, 42, 123)",0.912975,0.001778,0.917124,0.001501,0.913151,4.666667,534.937039,0.003973,0.002090,0.000000,1.000000
7,Reddit,sum,tgn_gru,3,"(0, 42, 123)",0.911373,0.023626,0.914302,0.019669,0.854119,5.000000,42.802287,0.060183,0.055943,0.000000,1.000000
8,Reddit,settransformer,tgn_gru,3,"(0, 42, 123)",0.906216,0.000984,0.905529,0.001309,0.904504,4.666667,105.481663,0.001025,0.001320,0.333333,1.000000
9,Reddit,hopfield,tgn_gru,3,"(0, 42, 123)",0.902289,0.003738,0.903005,0.002994,0.901769,5.666667,61.920998,0.001236,0.001290,0.000000,1.000000


Leaderboard shape: (99, 16)


,dataset,aggregator,update,n_completed,seed_list,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,mean_final_test_mrr,mean_best_epoch,mean_wall_sec,mean_overfit_gap,mean_val_drop,frac_peaked_early,completion_rate
0,LastFM,deepsets,hnn,3,"(0, 42, 123)",0.236944,0.079047,0.251735,0.091341,0.249777,4.333333,3278.854604,0.001958,0.001350,0.333333,1.0
1,LastFM,deepsets,hopfield_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,2918.614064,0.000000,0.000000,1.000000,1.0
2,LastFM,deepsets,ift_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,2492.739492,0.000000,0.000000,1.000000,1.0
3,LastFM,deepsets,lnn,3,"(0, 42, 123)",0.230624,0.002923,0.259484,0.008683,0.171725,2.000000,2664.337146,0.087759,0.052992,0.666667,1.0
4,LastFM,deepsets,tgn_gru,3,"(0, 42, 123)",0.389457,0.028856,0.412289,0.037406,0.384846,3.333333,2215.214454,0.027443,0.025476,0.333333,1.0
5,LastFM,hopfield,hnn,3,"(0, 42, 123)",0.300555,0.013996,0.337531,0.013822,0.309666,4.000000,3494.146113,0.027865,0.024177,0.000000,1.0
6,LastFM,hopfield,hopfield_update,3,"(0, 42, 123)",0.413990,0.011281,0.443000,0.010516,0.411214,3.333333,3964.570646,0.031786,0.029827,0.333333,1.0
7,LastFM,hopfield,lnn,3,"(0, 42, 123)",0.248813,0.022025,0.267674,0.014673,0.175598,3.666667,2980.186188,0.092075,0.078884,0.333333,1.0
8,LastFM,hopfield,tgn_gru,3,"(0, 42, 123)",0.397860,0.012964,0.427766,0.009140,0.420591,5.333333,2505.330206,0.007175,0.012661,0.000000,1.0
9,LastFM,ift,hnn,3,"(0, 42, 123)",0.343373,0.044620,0.368250,0.052612,0.345929,3.666667,3541.207378,0.022321,0.018104,0.333333,1.0


In [25]:
best_epoch_detail = (
    run_df.groupby(["dataset", "aggregator", "update"])["best_epoch"]
    .agg(
        best_epoch_list=lambda s: tuple(sorted(s.tolist())),
        mean_best_epoch="mean",
        std_best_epoch="std",
    )
    .reset_index()
)

leaderboard = leaderboard.drop(
    columns=[c for c in ["best_epoch_list", "mean_best_epoch", "std_best_epoch"] if c in leaderboard.columns],
    errors="ignore"
).merge(
    best_epoch_detail,
    on=["dataset", "aggregator", "update"],
    how="left"
)

display(leaderboard.head())

,dataset,aggregator,update,n_completed,seed_list,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,mean_final_test_mrr,mean_wall_sec,mean_overfit_gap,mean_val_drop,frac_peaked_early,completion_rate,best_epoch_list,mean_best_epoch,std_best_epoch
0,LastFM,deepsets,hnn,3,"(0, 42, 123)",0.236944,0.079047,0.251735,0.091341,0.249777,3278.854604,0.001958,0.001350,0.333333,1.0,"(2, 5, 6)",4.333333,2.081666
1,LastFM,deepsets,hopfield_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,2918.614064,0.000000,0.000000,1.000000,1.0,"(1, 1, 1)",1.000000,0.000000
2,LastFM,deepsets,ift_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,2492.739492,0.000000,0.000000,1.000000,1.0,"(1, 1, 1)",1.000000,0.000000
3,LastFM,deepsets,lnn,3,"(0, 42, 123)",0.230624,0.002923,0.259484,0.008683,0.171725,2664.337146,0.087759,0.052992,0.666667,1.0,"(1, 1, 4)",2.000000,1.732051
4,LastFM,deepsets,tgn_gru,3,"(0, 42, 123)",0.389457,0.028856,0.412289,0.037406,0.384846,2215.214454,0.027443,0.025476,0.333333,1.0,"(1, 3, 6)",3.333333,2.516611


In [26]:
# rank within each dataset first
leaderboard_ranked = leaderboard.copy()
leaderboard_ranked["val_rank_in_dataset"] = (
    leaderboard_ranked.groupby("dataset")["mean_best_val_mrr"]
    .rank(ascending=False, method="average")
)

cross_dataset = (
    leaderboard_ranked.groupby(["aggregator", "update"], dropna=False)
    .agg(
        datasets_seen=("dataset", "nunique"),
        mean_of_mean_best_val_mrr=("mean_best_val_mrr", "mean"),
        std_of_mean_best_val_mrr=("mean_best_val_mrr", "std"),
        mean_of_mean_best_test_mrr=("mean_best_test_mrr", "mean"),
        mean_completion_rate=("completion_rate", "mean"),
        mean_wall_sec=("mean_wall_sec", "mean"),
        mean_best_epoch=("mean_best_epoch", "mean"),
        mean_val_drop=("mean_val_drop", "mean"),
        mean_overfit_gap=("mean_overfit_gap", "mean"),
        top5_count=("val_rank_in_dataset", lambda s: (s <= 5).sum()),
        bottom5_count=("val_rank_in_dataset", lambda s: (s >= 21).sum()),  # 25 combos total
    )
    .reset_index()
    .sort_values(
        ["mean_of_mean_best_val_mrr", "mean_of_mean_best_test_mrr"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(cross_dataset)

,aggregator,update,datasets_seen,mean_of_mean_best_val_mrr,std_of_mean_best_val_mrr,mean_of_mean_best_test_mrr,mean_completion_rate,mean_wall_sec,mean_best_epoch,mean_val_drop,mean_overfit_gap,top5_count,bottom5_count
0,deepsets,hopfield_update,4,0.924795,0.072402,0.922725,1.000000,1022.819215,3.333333,0.005856,0.007457,1,0
1,deepsets,ift_update,4,0.916587,0.139302,0.921633,1.000000,658.397227,2.000000,0.004470,0.003201,2,0
2,settransformer,ift_update,4,0.891486,0.128333,0.897272,1.000000,1087.526922,3.250000,0.079438,0.079561,1,0
3,hopfield,ift_update,3,0.838466,0.132675,0.844613,0.777778,59.381473,2.444444,0.004378,0.001221,0,1
4,ift,tgn_gru,4,0.803491,0.246384,0.812606,1.000000,662.134598,4.416667,0.004799,0.005409,4,0
5,ift,hopfield_update,4,0.795577,0.245501,0.804142,1.000000,1098.050700,3.750000,0.032460,0.031369,3,0
6,sum,hopfield_update,4,0.792417,0.245506,0.798164,1.000000,994.376688,4.083333,0.027388,0.029122,2,0
7,sum,tgn_gru,4,0.788624,0.238873,0.793307,1.000000,525.668301,3.833333,0.042593,0.044852,2,0
8,hopfield,tgn_gru,4,0.771867,0.255552,0.777676,1.000000,664.754871,4.666667,0.005661,0.003952,0,0
9,deepsets,tgn_gru,4,0.771482,0.262134,0.771518,1.000000,584.713741,4.166667,0.010678,0.011094,1,0


In [27]:
print("=== Who peaks early most often? ===")
display(
    leaderboard.sort_values(["dataset", "frac_peaked_early", "mean_best_val_mrr"], ascending=[True, False, False])
)

print("=== Slow but good vs fast but mediocre ===")
display(
    leaderboard[[
        "dataset", "aggregator", "update",
        "mean_best_val_mrr", "mean_best_test_mrr",
        "mean_best_epoch", "mean_wall_sec",
        "completion_rate"
    ]].sort_values(["dataset", "mean_best_val_mrr"], ascending=[True, False])
)

print("=== Biggest val drop by combo ===")
display(
    leaderboard[[
        "dataset", "aggregator", "update",
        "mean_best_val_mrr", "mean_val_drop", "mean_overfit_gap",
        "frac_peaked_early"
    ]].sort_values(["dataset", "mean_val_drop"], ascending=[True, False])
)

=== Who peaks early most often? ===


,dataset,aggregator,update,n_completed,seed_list,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,mean_final_test_mrr,mean_wall_sec,mean_overfit_gap,mean_val_drop,frac_peaked_early,completion_rate,best_epoch_list,mean_best_epoch,std_best_epoch
1,LastFM,deepsets,hopfield_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,2918.614064,0.000000,0.000000,1.000000,1.0,"(1, 1, 1)",1.000000,0.000000
2,LastFM,deepsets,ift_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,1.000000,2492.739492,0.000000,0.000000,1.000000,1.0,"(1, 1, 1)",1.000000,0.000000
16,LastFM,settransformer,ift_update,3,"(0, 42, 123)",1.000000,0.000000,1.000000,0.000000,0.691122,4075.042972,0.308878,0.308001,1.000000,1.0,"(1, 1, 1)",1.000000,0.000000
23,LastFM,sum,tgn_gru,3,"(0, 42, 123)",0.438207,0.002694,0.461794,0.006641,0.427408,1995.287956,0.034385,0.041667,0.666667,1.0,"(2, 2, 5)",3.000000,1.732051
18,LastFM,settransformer,tgn_gru,3,"(0, 42, 123)",0.371546,0.009323,0.404226,0.003163,0.317833,3297.369553,0.086393,0.075893,0.666667,1.0,"(1, 1, 3)",1.666667,1.154701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,Wikipedia,hopfield,tgn_gru,3,"(0, 42, 123)",0.825241,0.001610,0.825930,0.000900,0.820692,45.966172,0.005238,0.004205,0.000000,1.0,"(4, 4, 6)",4.666667,1.154701
78,Wikipedia,deepsets,tgn_gru,3,"(0, 42, 123)",0.816450,0.016194,0.804569,0.022176,0.786839,37.859519,0.017730,0.013615,0.000000,1.0,"(4, 5, 6)",5.000000,1.000000
94,Wikipedia,sum,hnn,3,"(0, 42, 123)",0.690991,0.047615,0.713876,0.040083,0.701702,51.283665,0.012174,0.015581,0.000000,1.0,"(4, 5, 6)",5.000000,1.000000
77,Wikipedia,deepsets,lnn,3,"(0, 42, 123)",0.667632,0.002976,0.692939,0.001619,0.691302,43.083511,0.001637,0.002176,0.000000,1.0,"(3, 4, 6)",4.333333,1.527525


=== Slow but good vs fast but mediocre ===


,dataset,aggregator,update,mean_best_val_mrr,mean_best_test_mrr,mean_best_epoch,mean_wall_sec,completion_rate
1,LastFM,deepsets,hopfield_update,1.000000,1.000000,1.000000,2918.614064,1.0
2,LastFM,deepsets,ift_update,1.000000,1.000000,1.000000,2492.739492,1.0
16,LastFM,settransformer,ift_update,1.000000,1.000000,1.000000,4075.042972,1.0
13,LastFM,ift,tgn_gru,0.440791,0.475079,4.000000,2513.622913,1.0
23,LastFM,sum,tgn_gru,0.438207,0.461794,3.000000,1995.287956,1.0
...,...,...,...,...,...,...,...,...
89,Wikipedia,settransformer,hnn,0.672141,0.708114,2.666667,61.884812,1.0
77,Wikipedia,deepsets,lnn,0.667632,0.692939,4.333333,43.083511,1.0
97,Wikipedia,sum,lnn,0.667483,0.694547,3.333333,41.224728,1.0
82,Wikipedia,hopfield,lnn,0.654636,0.688217,2.333333,49.303586,1.0


=== Biggest val drop by combo ===


,dataset,aggregator,update,mean_best_val_mrr,mean_val_drop,mean_overfit_gap,frac_peaked_early
16,LastFM,settransformer,ift_update,1.000000,0.308001,0.308878,1.000000
22,LastFM,sum,lnn,0.216610,0.116951,0.147103,0.333333
20,LastFM,sum,hopfield_update,0.429999,0.082144,0.082428,0.333333
7,LastFM,hopfield,lnn,0.248813,0.078884,0.092075,0.333333
18,LastFM,settransformer,tgn_gru,0.371546,0.075893,0.086393,0.666667
...,...,...,...,...,...,...,...
76,Wikipedia,deepsets,ift_update,0.709918,0.002538,0.001426,0.333333
77,Wikipedia,deepsets,lnn,0.667632,0.002176,0.001637,0.000000
88,Wikipedia,ift,tgn_gru,0.860372,0.001761,0.003169,0.333333
92,Wikipedia,settransformer,lnn,0.642608,0.000929,0.002717,0.000000


In [28]:
# -----------------------------------
# Step 3: judge "uselessness"
# -----------------------------------

lb = leaderboard.copy()

NUM_COMBOS = len(EXPECTED_AGGS) * len(EXPECTED_UPDATES)  # 25

# rank within each dataset (1 = best)
lb["val_rank_in_dataset"] = (
    lb.groupby("dataset")["mean_best_val_mrr"]
      .rank(ascending=False, method="min")
)

lb["test_rank_in_dataset"] = (
    lb.groupby("dataset")["mean_best_test_mrr"]
      .rank(ascending=False, method="min")
)

# bucket by rank within dataset
def rank_bucket(rank, n=NUM_COMBOS):
    if rank <= 5:
        return "top_5"
    elif rank >= n - 4:   # bottom 5 out of 25 => ranks 21-25
        return "bottom_5"
    else:
        return "middle"

lb["val_rank_bucket"] = lb["val_rank_in_dataset"].apply(rank_bucket)
lb["test_rank_bucket"] = lb["test_rank_in_dataset"].apply(rank_bucket)

# -----------------------------------
# compare each combo to SUM baseline within same dataset
# -----------------------------------
sum_baseline = (
    lb[lb["aggregator"] == "sum"][[
        "dataset",
        "mean_best_val_mrr",
        "mean_best_test_mrr",
        "mean_wall_sec"
    ]]
    .rename(columns={
        "mean_best_val_mrr": "sum_best_val_mrr",
        "mean_best_test_mrr": "sum_best_test_mrr",
        "mean_wall_sec": "sum_wall_sec",
    })
)

lb = lb.merge(sum_baseline, on="dataset", how="left")

lb["vs_sum_val_diff"] = lb["mean_best_val_mrr"] - lb["sum_best_val_mrr"]
lb["vs_sum_test_diff"] = lb["mean_best_test_mrr"] - lb["sum_best_test_mrr"]
lb["runtime_ratio_vs_sum"] = lb["mean_wall_sec"] / lb["sum_wall_sec"]

# -----------------------------------
# heuristics / thresholds
# tweak these if you want stricter or looser rules
# -----------------------------------
VAL_DIFF_TOL = 0.01       # "meaningfully worse than sum" threshold
TEST_DIFF_TOL = 0.01
SLOW_RATIO = 1.5          # 50% slower than sum
HIGH_VAR_STD = 0.03       # seed std threshold (adjust if needed)
EARLY_PEAK_FRAC = 2/3     # if 2 of 3 seeds peak early on avg
OVERFIT_GAP_BAD = 0.01
VAL_DROP_BAD = 0.01

# safety: fill NaN stds for groups with only 1 completed seed
lb["std_best_val_mrr"] = lb["std_best_val_mrr"].fillna(0.0)
lb["std_best_test_mrr"] = lb["std_best_test_mrr"].fillna(0.0)

# -----------------------------------
# per-dataset flags
# -----------------------------------
lb["flag_incomplete"] = lb["completion_rate"] < 1.0
lb["flag_bottom5_val"] = lb["val_rank_in_dataset"] >= (NUM_COMBOS - 4)
lb["flag_bottom5_test"] = lb["test_rank_in_dataset"] >= (NUM_COMBOS - 4)

lb["flag_worse_than_sum_val"] = lb["vs_sum_val_diff"] < -VAL_DIFF_TOL
lb["flag_worse_than_sum_test"] = lb["vs_sum_test_diff"] < -TEST_DIFF_TOL
lb["flag_slower_than_sum"] = lb["runtime_ratio_vs_sum"] > SLOW_RATIO

lb["flag_high_val_var"] = lb["std_best_val_mrr"] > HIGH_VAR_STD
lb["flag_high_test_var"] = lb["std_best_test_mrr"] > HIGH_VAR_STD

lb["flag_peaks_early"] = lb["frac_peaked_early"] >= EARLY_PEAK_FRAC
lb["flag_overfit_gap"] = lb["mean_overfit_gap"] > OVERFIT_GAP_BAD
lb["flag_val_drop"] = lb["mean_val_drop"] > VAL_DROP_BAD

# count flags for convenience
flag_cols = [c for c in lb.columns if c.startswith("flag_")]
lb["n_flags"] = lb[flag_cols].sum(axis=1)

# -----------------------------------
# per-dataset verdict
# -----------------------------------
def verdict_row(row):
    # strong keep
    if (
        row["completion_rate"] == 1.0 and
        row["val_rank_in_dataset"] <= 5 and
        not row["flag_worse_than_sum_val"]
    ):
        return "strong_keep"

    # probably useless
    if (
        row["flag_incomplete"] or
        row["flag_bottom5_val"] or
        (row["flag_worse_than_sum_val"] and row["flag_worse_than_sum_test"]) or
        (row["flag_slower_than_sum"] and row["flag_worse_than_sum_val"]) or
        (row["n_flags"] >= 4)
    ):
        return "probably_useless"

    # otherwise
    return "maybe_keep"

lb["dataset_verdict"] = lb.apply(verdict_row, axis=1)

print("=== Per-dataset verdict counts ===")
display(lb["dataset_verdict"].value_counts().rename_axis("dataset_verdict").reset_index(name="count"))

print("=== Per-dataset flagged table ===")
display(
    lb[[
        "dataset", "aggregator", "update",
        "completion_rate",
        "val_rank_in_dataset", "test_rank_in_dataset",
        "mean_best_val_mrr", "std_best_val_mrr",
        "mean_best_test_mrr", "std_best_test_mrr",
        "mean_wall_sec", "runtime_ratio_vs_sum",
        "vs_sum_val_diff", "vs_sum_test_diff",
        "mean_best_epoch", "frac_peaked_early",
        "mean_val_drop", "mean_overfit_gap",
        "n_flags", "dataset_verdict"
    ]]
    .sort_values(["dataset", "dataset_verdict", "val_rank_in_dataset"], ascending=[True, True, True])
    .reset_index(drop=True)
)

=== Per-dataset verdict counts ===


,dataset_verdict,count
0,probably_useless,201
1,maybe_keep,196
2,strong_keep,98


=== Per-dataset flagged table ===


,dataset,aggregator,update,completion_rate,val_rank_in_dataset,test_rank_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,mean_wall_sec,runtime_ratio_vs_sum,vs_sum_val_diff,vs_sum_test_diff,mean_best_epoch,frac_peaked_early,mean_val_drop,mean_overfit_gap,n_flags,dataset_verdict
0,LastFM,ift,hopfield_update,1.0,6.0,5.0,0.433506,0.004482,0.465878,0.004632,3204.043232,1.096217,0.144965,0.150573,2.666667,0.333333,0.074732,0.082716,2,maybe_keep
1,LastFM,ift,hopfield_update,1.0,6.0,5.0,0.433506,0.004482,0.465878,0.004632,3204.043232,1.136714,0.003506,0.008805,2.666667,0.333333,0.074732,0.082716,2,maybe_keep
2,LastFM,ift,hopfield_update,1.0,6.0,5.0,0.433506,0.004482,0.465878,0.004632,3204.043232,1.298065,0.165034,0.161906,2.666667,0.333333,0.074732,0.082716,2,maybe_keep
3,LastFM,ift,hopfield_update,1.0,6.0,5.0,0.433506,0.004482,0.465878,0.004632,3204.043232,1.265848,0.216896,0.225897,2.666667,0.333333,0.074732,0.082716,2,maybe_keep
4,LastFM,ift,hopfield_update,1.0,6.0,5.0,0.433506,0.004482,0.465878,0.004632,3204.043232,1.605805,-0.004701,0.004084,2.666667,0.333333,0.074732,0.082716,3,maybe_keep
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,Wikipedia,settransformer,hopfield_update,1.0,4.0,5.0,0.841935,0.013559,0.839650,0.014128,447.884442,13.975551,0.000324,-0.000267,6.000000,0.000000,0.000000,0.000000,1,strong_keep
491,Wikipedia,sum,tgn_gru,1.0,5.0,4.0,0.841612,0.011573,0.839917,0.011380,32.047713,0.624911,0.150620,0.126041,3.333333,0.333333,0.060235,0.079266,2,strong_keep
492,Wikipedia,sum,tgn_gru,1.0,5.0,4.0,0.841612,0.011573,0.839917,0.011380,32.047713,0.751724,0.119938,0.099645,3.333333,0.333333,0.060235,0.079266,2,strong_keep
493,Wikipedia,sum,tgn_gru,1.0,5.0,4.0,0.841612,0.011573,0.839917,0.011380,32.047713,0.777391,0.174129,0.145370,3.333333,0.333333,0.060235,0.079266,2,strong_keep


In [29]:
# -----------------------------------
# Cross-dataset uselessness summary
# -----------------------------------

combo_judgment = (
    lb.groupby(["aggregator", "update"], dropna=False)
      .agg(
          datasets_seen=("dataset", "nunique"),

          mean_val_rank=("val_rank_in_dataset", "mean"),
          mean_test_rank=("test_rank_in_dataset", "mean"),

          mean_best_val_mrr=("mean_best_val_mrr", "mean"),
          mean_best_test_mrr=("mean_best_test_mrr", "mean"),

          mean_completion_rate=("completion_rate", "mean"),
          min_completion_rate=("completion_rate", "min"),

          mean_vs_sum_val_diff=("vs_sum_val_diff", "mean"),
          mean_vs_sum_test_diff=("vs_sum_test_diff", "mean"),
          mean_runtime_ratio_vs_sum=("runtime_ratio_vs_sum", "mean"),

          mean_wall_sec=("mean_wall_sec", "mean"),
          mean_best_epoch=("mean_best_epoch", "mean"),
          mean_frac_peaked_early=("frac_peaked_early", "mean"),
          mean_val_drop=("mean_val_drop", "mean"),
          mean_overfit_gap=("mean_overfit_gap", "mean"),

          top5_count=("val_rank_in_dataset", lambda s: (s <= 5).sum()),
          bottom5_count=("val_rank_in_dataset", lambda s: (s >= (NUM_COMBOS - 4)).sum()),

          incomplete_dataset_count=("flag_incomplete", "sum"),
          worse_than_sum_val_count=("flag_worse_than_sum_val", "sum"),
          worse_than_sum_test_count=("flag_worse_than_sum_test", "sum"),
          slower_and_worse_count=(
              "flag_slower_than_sum",
              lambda s: int(np.sum(s))
          ),
          useless_dataset_count=("dataset_verdict", lambda s: (s == "probably_useless").sum()),
          strong_dataset_count=("dataset_verdict", lambda s: (s == "strong_keep").sum()),
      )
      .reset_index()
)

# better slower+worse count: need both conditions at row level
slower_and_worse = (
    lb.assign(slower_and_worse=lambda d: d["flag_slower_than_sum"] & d["flag_worse_than_sum_val"])
      .groupby(["aggregator", "update"])["slower_and_worse"]
      .sum()
      .reset_index(name="slower_and_worse_count")
)

combo_judgment = combo_judgment.drop(columns=["slower_and_worse_count"]).merge(
    slower_and_worse,
    on=["aggregator", "update"],
    how="left"
)

def overall_combo_verdict(row):
    # strong keep: consistently good
    if (
        row["min_completion_rate"] == 1.0 and
        row["top5_count"] >= 2 and
        row["bottom5_count"] == 0 and
        row["mean_vs_sum_val_diff"] >= -0.005
    ):
        return "strong_keep"

    # probably useless: repeated weakness
    if (
        row["incomplete_dataset_count"] >= 2 or
        row["bottom5_count"] >= 2 or
        row["useless_dataset_count"] >= 2 or
        row["worse_than_sum_val_count"] >= 3 or
        row["slower_and_worse_count"] >= 2
    ):
        return "probably_useless"

    return "maybe_keep"

combo_judgment["overall_verdict"] = combo_judgment.apply(overall_combo_verdict, axis=1)

print("=== Overall combo verdict counts ===")
display(combo_judgment["overall_verdict"].value_counts().rename_axis("overall_verdict").reset_index(name="count"))

print("=== Cross-dataset combo judgment ===")
display(
    combo_judgment.sort_values(
        ["overall_verdict", "mean_val_rank", "mean_best_val_mrr"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
)

=== Overall combo verdict counts ===


,overall_verdict,count
0,probably_useless,13
1,strong_keep,12


=== Cross-dataset combo judgment ===


,aggregator,update,datasets_seen,mean_val_rank,mean_test_rank,mean_best_val_mrr,mean_best_test_mrr,mean_completion_rate,min_completion_rate,mean_vs_sum_val_diff,...,mean_overfit_gap,top5_count,bottom5_count,incomplete_dataset_count,worse_than_sum_val_count,worse_than_sum_test_count,useless_dataset_count,strong_dataset_count,slower_and_worse_count,overall_verdict
0,hopfield,tgn_gru,4,9.500000,10.500000,0.771867,0.777676,1.000000,1.000000,0.036371,...,0.003952,0,0,0,5,6,5,0,0,probably_useless
1,ift,ift_update,4,12.750000,9.250000,0.768340,0.783317,1.000000,1.000000,0.032844,...,0.003974,0,5,0,8,4,7,0,0,probably_useless
2,hopfield,hopfield_update,4,13.000000,13.500000,0.768917,0.774179,1.000000,1.000000,0.033422,...,0.022573,0,0,0,7,8,10,0,3,probably_useless
3,sum,ift_update,4,14.500000,14.750000,0.710539,0.721925,1.000000,1.000000,-0.024957,...,0.006652,0,0,0,7,7,7,0,0,probably_useless
4,sum,hnn,4,15.750000,16.250000,0.706510,0.716739,1.000000,1.000000,-0.028986,...,0.006364,0,0,0,7,7,12,0,1,probably_useless
5,deepsets,hnn,4,16.750000,18.500000,0.699530,0.705886,1.000000,1.000000,-0.035965,...,0.008488,0,5,0,8,8,9,0,3,probably_useless
6,hopfield,hnn,4,19.000000,19.000000,0.703906,0.717409,1.000000,1.000000,-0.031590,...,0.014150,0,5,0,10,9,11,0,3,probably_useless
7,settransformer,hnn,4,19.250000,18.000000,0.697223,0.711488,1.000000,1.000000,-0.038273,...,0.007589,0,10,0,12,10,17,0,4,probably_useless
8,hopfield,ift_update,3,20.333333,20.666667,0.838466,0.844613,0.777778,0.333333,-0.032740,...,0.001221,0,5,5,10,10,10,0,3,probably_useless
9,sum,lnn,4,21.250000,21.250000,0.679389,0.689258,1.000000,1.000000,-0.056107,...,0.038324,0,15,0,14,14,17,0,0,probably_useless


In [35]:
import math
import pandas as pd
import numpy as np

# make a working copy
lb = leaderboard.copy()

# -----------------------------
# helper: assign thirds by rank
# -----------------------------
def assign_thirds(df, metric_col):
    """
    Rank descending by metric_col within a dataset and assign:
    top_third / middle_third / bottom_third
    """
    df = df.sort_values(metric_col, ascending=False).reset_index(drop=True).copy()
    n = len(df)

    # ranks 1..n
    df["rank"] = np.arange(1, n + 1)

    # split sizes
    top_cut = math.ceil(n / 3)
    mid_cut = math.ceil(2 * n / 3)

    def bucket(rank):
        if rank <= top_cut:
            return "top_third"
        elif rank <= mid_cut:
            return "middle_third"
        else:
            return "bottom_third"

    df["third_bucket"] = df["rank"].apply(bucket)
    return df


# -----------------------------
# VAL TABLES
# -----------------------------
val_top_thirds = {}

for ds in EXPECTED_DATASETS:
    ds_df = lb[lb["dataset"] == ds].copy()

    # metric range
    val_min = ds_df["mean_best_val_mrr"].min()
    val_max = ds_df["mean_best_val_mrr"].max()
    val_range = val_max - val_min

    ranked = assign_thirds(ds_df, "mean_best_val_mrr")
    ranked["val_range_min"] = val_min
    ranked["val_range_max"] = val_max
    ranked["val_range_width"] = val_range

    top_third = ranked[ranked["third_bucket"] == "top_third"].copy()

    top_third = top_third[[
        "dataset",
        "aggregator",
        "update",
        "rank",
        "third_bucket",
        "mean_best_val_mrr",
        "best_epoch_list",
        "std_best_epoch",
        "std_best_val_mrr",
        "mean_best_test_mrr",
        "mean_wall_sec",
        "mean_best_epoch",
        "completion_rate",
        "val_range_min",
        "val_range_max",
        "val_range_width",
    ]].sort_values("rank")

    val_top_thirds[ds] = top_third

    print(f"\n=== {ds} : VALIDATION MRR ===")
    print(f"range: min={val_min:.6f}, max={val_max:.6f}, width={val_range:.6f}")
    display(top_third.reset_index(drop=True))

best_epoch_detail = (
    run_df.groupby(["dataset", "aggregator", "update"])["best_epoch"]
    .agg(
        best_epoch_list=lambda s: tuple(sorted(s.tolist())),
        mean_best_epoch="mean",
        std_best_epoch="std",
    )
    .reset_index()
)
# -----------------------------
# TEST TABLES
# -----------------------------
test_top_thirds = {}

for ds in EXPECTED_DATASETS:
    ds_df = lb[lb["dataset"] == ds].copy()

    # metric range
    test_min = ds_df["mean_best_test_mrr"].min()
    test_max = ds_df["mean_best_test_mrr"].max()
    test_range = test_max - test_min

    ranked = assign_thirds(ds_df, "mean_best_test_mrr")
    ranked["test_range_min"] = test_min
    ranked["test_range_max"] = test_max
    ranked["test_range_width"] = test_range

    top_third = ranked[ranked["third_bucket"] == "top_third"].copy()

    top_third = top_third[[
        "dataset",
        "aggregator",
        "update",
        "rank",
        "third_bucket",
        "mean_best_test_mrr",
        "std_best_test_mrr",
        "mean_best_val_mrr",
        "mean_wall_sec",
        "mean_best_epoch",
        "best_epoch_list",
        "completion_rate",
        "std_best_epoch",
        "test_range_min",
        "test_range_max",
        "test_range_width",
    ]].sort_values("rank")

    test_top_thirds[ds] = top_third

    print(f"\n=== {ds} : TEST MRR ===")
    print(f"range: min={test_min:.6f}, max={test_max:.6f}, width={test_range:.6f}")
    display(top_third.reset_index(drop=True))

    


=== Wikipedia : VALIDATION MRR ===
range: min=0.642608, max=0.860372, width=0.217765


,dataset,aggregator,update,rank,third_bucket,mean_best_val_mrr,best_epoch_list,std_best_epoch,std_best_val_mrr,mean_best_test_mrr,mean_wall_sec,mean_best_epoch,completion_rate,val_range_min,val_range_max,val_range_width
0,Wikipedia,ift,tgn_gru,1,top_third,0.860372,"(2, 4, 6)",2.000000,0.001454,0.866016,43.048286,4.000000,1.0,0.642608,0.860372,0.217765
1,Wikipedia,sum,hopfield_update,2,top_third,0.860003,"(2, 3, 6)",2.081666,0.001474,0.855444,389.023909,3.666667,1.0,0.642608,0.860372,0.217765
2,Wikipedia,ift,hopfield_update,3,top_third,0.857577,"(4, 5, 6)",1.000000,0.003452,0.863980,397.996660,5.000000,1.0,0.642608,0.860372,0.217765
3,Wikipedia,settransformer,hopfield_update,4,top_third,0.841935,"(6, 6, 6)",0.000000,0.013559,0.839650,447.884442,6.000000,1.0,0.642608,0.860372,0.217765
4,Wikipedia,sum,tgn_gru,5,top_third,0.841612,"(2, 3, 5)",1.527525,0.011573,0.839917,32.047713,3.333333,1.0,0.642608,0.860372,0.217765
5,Wikipedia,settransformer,tgn_gru,6,top_third,0.837348,"(4, 5, 6)",1.000000,0.012149,0.837820,67.174628,5.000000,1.0,0.642608,0.860372,0.217765
6,Wikipedia,deepsets,hopfield_update,7,top_third,0.837123,"(4, 4, 6)",1.154701,0.005219,0.833010,393.655440,4.666667,1.0,0.642608,0.860372,0.217765
7,Wikipedia,hopfield,tgn_gru,8,top_third,0.825241,"(4, 4, 6)",1.154701,0.001610,0.825930,45.966172,4.666667,1.0,0.642608,0.860372,0.217765
8,Wikipedia,hopfield,hopfield_update,9,top_third,0.822390,"(2, 2, 6)",2.309401,0.008600,0.820832,425.870684,3.333333,1.0,0.642608,0.860372,0.217765



=== Reddit : VALIDATION MRR ===
range: min=0.867068, max=1.000000, width=0.132932


,dataset,aggregator,update,rank,third_bucket,mean_best_val_mrr,best_epoch_list,std_best_epoch,std_best_val_mrr,mean_best_test_mrr,mean_wall_sec,mean_best_epoch,completion_rate,val_range_min,val_range_max,val_range_width
0,Reddit,deepsets,ift_update,1,top_third,1.000000,"(2, 2, 2)",0.000000,0.000000,1.000000,54.240344,2.000000,1.0,0.867068,1.0,0.132932
1,Reddit,ift,tgn_gru,2,top_third,0.939561,"(5, 5, 6)",0.577350,0.003510,0.940129,50.313569,5.333333,1.0,0.867068,1.0,0.132932
2,Reddit,ift,hopfield_update,3,top_third,0.924985,"(3, 5, 5)",1.154701,0.005281,0.926777,544.409674,4.333333,1.0,0.867068,1.0,0.132932
3,Reddit,ift,hnn,4,top_third,0.920888,"(2, 2, 5)",1.732051,0.006690,0.923994,69.798048,3.000000,1.0,0.867068,1.0,0.132932
4,Reddit,ift,lnn,5,top_third,0.915876,"(6, 6, 6)",0.000000,0.001166,0.919047,60.554952,6.000000,1.0,0.867068,1.0,0.132932
5,Reddit,deepsets,tgn_gru,6,top_third,0.913801,"(4, 5, 6)",1.000000,0.005084,0.912834,48.917371,5.000000,1.0,0.867068,1.0,0.132932
6,Reddit,sum,hopfield_update,7,top_third,0.912975,"(4, 4, 6)",1.154701,0.001778,0.917124,534.937039,4.666667,1.0,0.867068,1.0,0.132932
7,Reddit,sum,tgn_gru,8,top_third,0.911373,"(4, 5, 6)",1.000000,0.023626,0.914302,42.802287,5.000000,1.0,0.867068,1.0,0.132932
8,Reddit,settransformer,tgn_gru,9,top_third,0.906216,"(2, 6, 6)",2.309401,0.000984,0.905529,105.481663,4.666667,1.0,0.867068,1.0,0.132932



=== MOOC : VALIDATION MRR ===
range: min=0.923105, max=0.973237, width=0.050132


...

,dataset,aggregator,update,rank,third_bucket,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,completion_rate,std_best_epoch,test_range_min,test_range_max,test_range_width
0,Reddit,deepsets,ift_update,1,top_third,1.000000,0.000000,1.000000,54.240344,2.000000,"(2, 2, 2)",1.0,0.000000,0.870665,1.0,0.129335
1,Reddit,ift,tgn_gru,2,top_third,0.940129,0.003214,0.939561,50.313569,5.333333,"(5, 5, 6)",1.0,0.577350,0.870665,1.0,0.129335
2,Reddit,ift,hopfield_update,3,top_third,0.926777,0.003281,0.924985,544.409674,4.333333,"(3, 5, 5)",1.0,1.154701,0.870665,1.0,0.129335
3,Reddit,ift,hnn,4,top_third,0.923994,0.005208,0.920888,69.798048,3.000000,"(2, 2, 5)",1.0,1.732051,0.870665,1.0,0.129335
4,Reddit,ift,lnn,5,top_third,0.919047,0.001441,0.915876,60.554952,6.000000,"(6, 6, 6)",1.0,0.000000,0.870665,1.0,0.129335
5,Reddit,sum,hopfield_update,6,top_third,0.917124,0.001501,0.912975,534.937039,4.666667,"(4, 4, 6)",1.0,1.154701,0.870665,1.0,0.129335
6,Reddit,sum,tgn_gru,7,top_third,0.914302,0.019669,0.911373,42.802287,5.000000,"(4, 5, 6)",1.0,1.000000,0.870665,1.0,0.129335
7,Reddit,deepsets,tgn_gru,8,top_third,0.912834,0.004072,0.913801,48.917371,5.000000,"(4, 5, 6)",1.0,1.000000,0.870665,1.0,0.129335
8,Reddit,settransformer,tgn_gru,9,top_third,0.905529,0.001309,0.906216,105.481663,4.666667,"(2, 6, 6)",1.0,2.309401,0.870665,1.0,0.129335



=== MOOC : TEST MRR ===
range: min=0.926285, max=0.969200, width=0.042915


,dataset,aggregator,update,rank,third_bucket,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,completion_rate,std_best_epoch,test_range_min,test_range_max,test_range_width
0,MOOC,ift,tgn_gru,1,top_third,0.969200,0.011324,0.973237,41.553625,4.333333,"(2, 5, 6)",1.0,2.081666,0.926285,0.9692,0.042915
1,MOOC,sum,hopfield_update,2,top_third,0.963014,0.006783,0.966691,234.856574,4.333333,"(3, 5, 5)",1.0,1.154701,0.926285,0.9692,0.042915
2,MOOC,settransformer,tgn_gru,3,top_third,0.960271,0.000391,0.966999,66.540998,3.333333,"(2, 3, 5)",1.0,1.527525,0.926285,0.9692,0.042915
3,MOOC,ift,hopfield_update,4,top_third,0.959932,0.007883,0.966239,245.753235,3.000000,"(1, 2, 6)",1.0,2.645751,0.926285,0.9692,0.042915
4,MOOC,deepsets,hopfield_update,5,top_third,0.958814,0.000970,0.964985,238.213580,3.666667,"(2, 4, 5)",1.0,1.527525,0.926285,0.9692,0.042915
5,MOOC,ift,ift_update,6,top_third,0.958428,0.001274,0.964978,45.744883,3.333333,"(2, 4, 4)",1.0,1.154701,0.926285,0.9692,0.042915
6,MOOC,sum,tgn_gru,7,top_third,0.957214,0.003726,0.963305,32.535249,4.000000,"(3, 4, 5)",1.0,1.000000,0.926285,0.9692,0.042915
7,MOOC,deepsets,tgn_gru,8,top_third,0.956380,0.006551,0.966221,36.863621,3.333333,"(1, 3, 6)",1.0,2.516611,0.926285,0.9692,0.042915
8,MOOC,sum,hnn,9,top_third,0.955377,0.000925,0.962930,51.411779,3.000000,"(2, 3, 4)",1.0,1.000000,0.926285,0.9692,0.042915



=== LastFM : TEST MRR ===
range: min=0.229785, max=1.000000, width=0.770215


,dataset,aggregator,update,rank,third_bucket,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,completion_rate,std_best_epoch,test_range_min,test_range_max,test_range_width
0,LastFM,deepsets,hopfield_update,1,top_third,1.000000,0.000000,1.000000,2918.614064,1.000000,"(1, 1, 1)",1.0,0.000000,0.229785,1.0,0.770215
1,LastFM,deepsets,ift_update,2,top_third,1.000000,0.000000,1.000000,2492.739492,1.000000,"(1, 1, 1)",1.0,0.000000,0.229785,1.0,0.770215
2,LastFM,settransformer,ift_update,3,top_third,1.000000,0.000000,1.000000,4075.042972,1.000000,"(1, 1, 1)",1.0,0.000000,0.229785,1.0,0.770215
3,LastFM,ift,tgn_gru,4,top_third,0.475079,0.002489,0.440791,2513.622913,4.000000,"(2, 4, 6)",1.0,2.000000,0.229785,1.0,0.770215
4,LastFM,ift,hopfield_update,5,top_third,0.465878,0.004632,0.433506,3204.043232,2.666667,"(2, 3, 3)",1.0,0.577350,0.229785,1.0,0.770215
5,LastFM,ift,ift_update,6,top_third,0.462959,0.000596,0.425462,2744.502441,4.333333,"(3, 5, 5)",1.0,1.154701,0.229785,1.0,0.770215
6,LastFM,sum,tgn_gru,7,top_third,0.461794,0.006641,0.438207,1995.287956,3.000000,"(2, 2, 5)",1.0,1.732051,0.229785,1.0,0.770215
7,LastFM,sum,hopfield_update,8,top_third,0.457073,0.010326,0.429999,2818.689230,3.666667,"(2, 4, 5)",1.0,1.527525,0.229785,1.0,0.770215


In [36]:
for ds in EXPECTED_DATASETS:
    print(f"\n=== {ds} : TOP THIRD BY VAL MRR ===")
    display(
        val_top_thirds[ds][[
            "aggregator",
            "update",
            "mean_best_val_mrr",
            "mean_wall_sec",
            "mean_best_epoch",
            "best_epoch_list",
            "std_best_epoch"
        ]]
    )

for ds in EXPECTED_DATASETS:
    print(f"\n=== {ds} : TOP THIRD BY TEST MRR ===")
    display(
        test_top_thirds[ds][[
            "aggregator",
            "update",
            "mean_best_test_mrr",
            "mean_wall_sec",
            "mean_best_epoch",
            "best_epoch_list",
            "std_best_epoch"
        ]]
    )


=== Wikipedia : TOP THIRD BY VAL MRR ===


,aggregator,update,mean_best_val_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,std_best_epoch
0,ift,tgn_gru,0.860372,43.048286,4.000000,"(2, 4, 6)",2.000000
1,sum,hopfield_update,0.860003,389.023909,3.666667,"(2, 3, 6)",2.081666
2,ift,hopfield_update,0.857577,397.996660,5.000000,"(4, 5, 6)",1.000000
3,settransformer,hopfield_update,0.841935,447.884442,6.000000,"(6, 6, 6)",0.000000
4,sum,tgn_gru,0.841612,32.047713,3.333333,"(2, 3, 5)",1.527525
5,settransformer,tgn_gru,0.837348,67.174628,5.000000,"(4, 5, 6)",1.000000
6,deepsets,hopfield_update,0.837123,393.655440,4.666667,"(4, 4, 6)",1.154701
7,hopfield,tgn_gru,0.825241,45.966172,4.666667,"(4, 4, 6)",1.154701
8,hopfield,hopfield_update,0.822390,425.870684,3.333333,"(2, 2, 6)",2.309401



=== Reddit : TOP THIRD BY VAL MRR ===


,aggregator,update,mean_best_val_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,std_best_epoch
0,deepsets,ift_update,1.000000,54.240344,2.000000,"(2, 2, 2)",0.000000
1,ift,tgn_gru,0.939561,50.313569,5.333333,"(5, 5, 6)",0.577350
2,ift,hopfield_update,0.924985,544.409674,4.333333,"(3, 5, 5)",1.154701
3,ift,hnn,0.920888,69.798048,3.000000,"(2, 2, 5)",1.732051
4,ift,lnn,0.915876,60.554952,6.000000,"(6, 6, 6)",0.000000
5,deepsets,tgn_gru,0.913801,48.917371,5.000000,"(4, 5, 6)",1.000000
6,sum,hopfield_update,0.912975,534.937039,4.666667,"(4, 4, 6)",1.154701
7,sum,tgn_gru,0.911373,42.802287,5.000000,"(4, 5, 6)",1.000000
8,settransformer,tgn_gru,0.906216,105.481663,4.666667,"(2, 6, 6)",2.309401



=== MOOC : TOP THIRD BY VAL MRR ===


...

,aggregator,update,mean_best_test_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,std_best_epoch
0,deepsets,ift_update,1.000000,54.240344,2.000000,"(2, 2, 2)",0.000000
1,ift,tgn_gru,0.940129,50.313569,5.333333,"(5, 5, 6)",0.577350
2,ift,hopfield_update,0.926777,544.409674,4.333333,"(3, 5, 5)",1.154701
3,ift,hnn,0.923994,69.798048,3.000000,"(2, 2, 5)",1.732051
4,ift,lnn,0.919047,60.554952,6.000000,"(6, 6, 6)",0.000000
5,sum,hopfield_update,0.917124,534.937039,4.666667,"(4, 4, 6)",1.154701
6,sum,tgn_gru,0.914302,42.802287,5.000000,"(4, 5, 6)",1.000000
7,deepsets,tgn_gru,0.912834,48.917371,5.000000,"(4, 5, 6)",1.000000
8,settransformer,tgn_gru,0.905529,105.481663,4.666667,"(2, 6, 6)",2.309401



=== MOOC : TOP THIRD BY TEST MRR ===


,aggregator,update,mean_best_test_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,std_best_epoch
0,ift,tgn_gru,0.969200,41.553625,4.333333,"(2, 5, 6)",2.081666
1,sum,hopfield_update,0.963014,234.856574,4.333333,"(3, 5, 5)",1.154701
2,settransformer,tgn_gru,0.960271,66.540998,3.333333,"(2, 3, 5)",1.527525
3,ift,hopfield_update,0.959932,245.753235,3.000000,"(1, 2, 6)",2.645751
4,deepsets,hopfield_update,0.958814,238.213580,3.666667,"(2, 4, 5)",1.527525
5,ift,ift_update,0.958428,45.744883,3.333333,"(2, 4, 4)",1.154701
6,sum,tgn_gru,0.957214,32.535249,4.000000,"(3, 4, 5)",1.000000
7,deepsets,tgn_gru,0.956380,36.863621,3.333333,"(1, 3, 6)",2.516611
8,sum,hnn,0.955377,51.411779,3.000000,"(2, 3, 4)",1.000000



=== LastFM : TOP THIRD BY TEST MRR ===


,aggregator,update,mean_best_test_mrr,mean_wall_sec,mean_best_epoch,best_epoch_list,std_best_epoch
0,deepsets,hopfield_update,1.000000,2918.614064,1.000000,"(1, 1, 1)",0.000000
1,deepsets,ift_update,1.000000,2492.739492,1.000000,"(1, 1, 1)",0.000000
2,settransformer,ift_update,1.000000,4075.042972,1.000000,"(1, 1, 1)",0.000000
3,ift,tgn_gru,0.475079,2513.622913,4.000000,"(2, 4, 6)",2.000000
4,ift,hopfield_update,0.465878,3204.043232,2.666667,"(2, 3, 3)",0.577350
5,ift,ift_update,0.462959,2744.502441,4.333333,"(3, 5, 5)",1.154701
6,sum,tgn_gru,0.461794,1995.287956,3.000000,"(2, 2, 5)",1.732051
7,sum,hopfield_update,0.457073,2818.689230,3.666667,"(2, 4, 5)",1.527525


In [38]:
# -----------------------------------
# IFT-involved models only
# "uses IFT in any capacity" means:
#   aggregator == "ift" OR update == "ift_update"
# -----------------------------------

ift_models = leaderboard[
    (leaderboard["aggregator"] == "ift") | (leaderboard["update"] == "ift_update")
].copy()

ift_val_tables = {}

for ds in EXPECTED_DATASETS:
    ds_df = ift_models[ift_models["dataset"] == ds].copy()

    ds_df = ds_df.sort_values(
        ["mean_best_val_mrr", "mean_best_test_mrr"],
        ascending=[False, False]
    ).reset_index(drop=True)

    ds_df["val_rank_among_ift"] = np.arange(1, len(ds_df) + 1)

    val_min = ds_df["mean_best_val_mrr"].min()
    val_max = ds_df["mean_best_val_mrr"].max()
    val_range = val_max - val_min

    ds_df["val_range_min"] = val_min
    ds_df["val_range_max"] = val_max
    ds_df["val_range_width"] = val_range

    out = ds_df[[
        "dataset",
        "aggregator",
        "update",
        "val_rank_among_ift",
        "mean_best_val_mrr",
        "std_best_val_mrr",
        "mean_best_test_mrr",
        "std_best_test_mrr",
        "best_epoch_list",
        "mean_best_epoch",
        "std_best_epoch",
        "mean_wall_sec",
        "completion_rate",
        "val_range_min",
        "val_range_max",
        "val_range_width",
    ]].copy()

    ift_val_tables[ds] = out

    print(f"\n=== {ds} : IFT-involved models ranked by VALIDATION MRR ===")
    print(f"val mrr range: min={val_min:.6f}, max={val_max:.6f}, width={val_range:.6f}")
    display(out)


=== Wikipedia : IFT-involved models ranked by VALIDATION MRR ===
val mrr range: min=0.693130, max=0.860372, width=0.167243


,dataset,aggregator,update,val_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,val_range_min,val_range_max,val_range_width
0,Wikipedia,ift,tgn_gru,1,0.860372,0.001454,0.866016,0.002074,"(2, 4, 6)",4.000000,2.000000,43.048286,1.0,0.69313,0.860372,0.167243
1,Wikipedia,ift,hopfield_update,2,0.857577,0.003452,0.863980,0.002124,"(4, 5, 6)",5.000000,1.000000,397.996660,1.0,0.69313,0.860372,0.167243
2,Wikipedia,ift,ift_update,3,0.815851,0.001271,0.828377,0.002290,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0,0.69313,0.860372,0.167243
3,Wikipedia,ift,hnn,4,0.800154,0.006924,0.817659,0.010110,"(1, 1, 3)",1.666667,1.154701,57.269979,1.0,0.69313,0.860372,0.167243
4,Wikipedia,ift,lnn,5,0.781483,0.002939,0.809615,0.002388,"(1, 2, 4)",2.333333,1.527525,49.873984,1.0,0.69313,0.860372,0.167243
5,Wikipedia,sum,ift_update,6,0.721674,0.002442,0.740272,0.001980,"(1, 1, 1)",1.000000,0.000000,42.632306,1.0,0.69313,0.860372,0.167243
6,Wikipedia,deepsets,ift_update,7,0.709918,0.001802,0.732147,0.001775,"(1, 4, 6)",3.666667,2.516611,44.296756,1.0,0.69313,0.860372,0.167243
7,Wikipedia,settransformer,ift_update,8,0.709603,0.000825,0.738808,0.000770,"(1, 4, 4)",3.000000,1.732051,80.028924,1.0,0.69313,0.860372,0.167243
8,Wikipedia,hopfield,ift_update,9,0.693130,0.003085,0.717308,0.003314,"(1, 1, 6)",2.666667,2.886751,53.113788,1.0,0.69313,0.860372,0.167243



=== Reddit : IFT-involved models ranked by VALIDATION MRR ===
val mrr range: min=0.867068, max=1.000000, width=0.132932


,dataset,aggregator,update,val_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,val_range_min,val_range_max,val_range_width
0,Reddit,deepsets,ift_update,1,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000,0.867068,1.0,0.132932
1,Reddit,ift,tgn_gru,2,0.939561,0.003510,0.940129,0.003214,"(5, 5, 6)",5.333333,0.577350,50.313569,1.000000,0.867068,1.0,0.132932
2,Reddit,ift,hopfield_update,3,0.924985,0.005281,0.926777,0.003281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.000000,0.867068,1.0,0.132932
3,Reddit,ift,hnn,4,0.920888,0.006690,0.923994,0.005208,"(2, 2, 5)",3.000000,1.732051,69.798048,1.000000,0.867068,1.0,0.132932
4,Reddit,ift,lnn,5,0.915876,0.001166,0.919047,0.001441,"(6, 6, 6)",6.000000,0.000000,60.554952,1.000000,0.867068,1.0,0.132932
5,Reddit,settransformer,ift_update,6,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000,0.867068,1.0,0.132932
6,Reddit,sum,ift_update,7,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000,0.867068,1.0,0.132932
7,Reddit,hopfield,ift_update,8,0.869175,NaN,0.870665,NaN,"(2,)",2.000000,NaN,71.289740,0.333333,0.867068,1.0,0.132932
8,Reddit,ift,ift_update,9,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.577350,59.139892,1.000000,0.867068,1.0,0.132932



=== MOOC : IFT-involved models ranked by VALIDATION MRR ===
val mrr range: min=0.953094, max=0.973237, width=0.020143


,dataset,aggregator,update,val_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,val_range_min,val_range_max,val_range_width
0,MOOC,ift,tgn_gru,1,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0,0.953094,0.973237,0.020143
1,MOOC,ift,hopfield_update,2,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0,0.953094,0.973237,0.020143
2,MOOC,ift,ift_update,3,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0,0.953094,0.973237,0.020143
3,MOOC,sum,ift_update,4,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.0,0.953094,0.973237,0.020143
4,MOOC,settransformer,ift_update,5,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.0,0.953094,0.973237,0.020143
5,MOOC,deepsets,ift_update,6,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.0,0.953094,0.973237,0.020143
6,MOOC,ift,hnn,7,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0,0.953094,0.973237,0.020143
7,MOOC,ift,lnn,8,0.953411,0.002691,0.940166,0.023555,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0,0.953094,0.973237,0.020143
8,MOOC,hopfield,ift_update,9,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.0,0.953094,0.973237,0.020143



=== LastFM : IFT-involved models ranked by VALIDATION MRR ===
val mrr range: min=0.268471, max=1.000000, width=0.731529


,dataset,aggregator,update,val_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,val_range_min,val_range_max,val_range_width
0,LastFM,deepsets,ift_update,1,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.0,0.268471,1.0,0.731529
1,LastFM,settransformer,ift_update,2,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.0,0.268471,1.0,0.731529
2,LastFM,ift,tgn_gru,3,0.440791,0.005729,0.475079,0.002489,"(2, 4, 6)",4.000000,2.000000,2513.622913,1.0,0.268471,1.0,0.731529
3,LastFM,ift,hopfield_update,4,0.433506,0.004482,0.465878,0.004632,"(2, 3, 3)",2.666667,0.577350,3204.043232,1.0,0.268471,1.0,0.731529
4,LastFM,ift,ift_update,5,0.425462,0.000290,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0,0.268471,1.0,0.731529
5,LastFM,ift,hnn,6,0.343373,0.044620,0.368250,0.052612,"(2, 4, 5)",3.666667,1.527525,3541.207378,1.0,0.268471,1.0,0.731529
6,LastFM,ift,lnn,7,0.330548,0.035811,0.350109,0.039379,"(5, 6, 6)",5.666667,0.577350,3022.797079,1.0,0.268471,1.0,0.731529
7,LastFM,sum,ift_update,8,0.268471,0.000587,0.303972,0.000744,"(4, 6, 6)",5.333333,1.154701,2468.323378,1.0,0.268471,1.0,0.731529


In [39]:
ift_test_tables = {}

for ds in EXPECTED_DATASETS:
    ds_df = ift_models[ift_models["dataset"] == ds].copy()

    ds_df = ds_df.sort_values(
        ["mean_best_test_mrr", "mean_best_val_mrr"],
        ascending=[False, False]
    ).reset_index(drop=True)

    ds_df["test_rank_among_ift"] = np.arange(1, len(ds_df) + 1)

    test_min = ds_df["mean_best_test_mrr"].min()
    test_max = ds_df["mean_best_test_mrr"].max()
    test_range = test_max - test_min

    ds_df["test_range_min"] = test_min
    ds_df["test_range_max"] = test_max
    ds_df["test_range_width"] = test_range

    out = ds_df[[
        "dataset",
        "aggregator",
        "update",
        "test_rank_among_ift",
        "mean_best_test_mrr",
        "std_best_test_mrr",
        "mean_best_val_mrr",
        "std_best_val_mrr",
        "best_epoch_list",
        "mean_best_epoch",
        "std_best_epoch",
        "mean_wall_sec",
        "completion_rate",
        "test_range_min",
        "test_range_max",
        "test_range_width",
    ]].copy()

    ift_test_tables[ds] = out

    print(f"\n=== {ds} : IFT-involved models ranked by TEST MRR ===")
    print(f"test mrr range: min={test_min:.6f}, max={test_max:.6f}, width={test_range:.6f}")
    display(out)


=== Wikipedia : IFT-involved models ranked by TEST MRR ===
test mrr range: min=0.717308, max=0.866016, width=0.148708


,dataset,aggregator,update,test_rank_among_ift,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,std_best_val_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,test_range_min,test_range_max,test_range_width
0,Wikipedia,ift,tgn_gru,1,0.866016,0.002074,0.860372,0.001454,"(2, 4, 6)",4.000000,2.000000,43.048286,1.0,0.717308,0.866016,0.148708
1,Wikipedia,ift,hopfield_update,2,0.863980,0.002124,0.857577,0.003452,"(4, 5, 6)",5.000000,1.000000,397.996660,1.0,0.717308,0.866016,0.148708
2,Wikipedia,ift,ift_update,3,0.828377,0.002290,0.815851,0.001271,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0,0.717308,0.866016,0.148708
3,Wikipedia,ift,hnn,4,0.817659,0.010110,0.800154,0.006924,"(1, 1, 3)",1.666667,1.154701,57.269979,1.0,0.717308,0.866016,0.148708
4,Wikipedia,ift,lnn,5,0.809615,0.002388,0.781483,0.002939,"(1, 2, 4)",2.333333,1.527525,49.873984,1.0,0.717308,0.866016,0.148708
5,Wikipedia,sum,ift_update,6,0.740272,0.001980,0.721674,0.002442,"(1, 1, 1)",1.000000,0.000000,42.632306,1.0,0.717308,0.866016,0.148708
6,Wikipedia,settransformer,ift_update,7,0.738808,0.000770,0.709603,0.000825,"(1, 4, 4)",3.000000,1.732051,80.028924,1.0,0.717308,0.866016,0.148708
7,Wikipedia,deepsets,ift_update,8,0.732147,0.001775,0.709918,0.001802,"(1, 4, 6)",3.666667,2.516611,44.296756,1.0,0.717308,0.866016,0.148708
8,Wikipedia,hopfield,ift_update,9,0.717308,0.003314,0.693130,0.003085,"(1, 1, 6)",2.666667,2.886751,53.113788,1.0,0.717308,0.866016,0.148708



=== Reddit : IFT-involved models ranked by TEST MRR ===
test mrr range: min=0.870665, max=1.000000, width=0.129335


,dataset,aggregator,update,test_rank_among_ift,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,std_best_val_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,test_range_min,test_range_max,test_range_width
0,Reddit,deepsets,ift_update,1,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000,0.870665,1.0,0.129335
1,Reddit,ift,tgn_gru,2,0.940129,0.003214,0.939561,0.003510,"(5, 5, 6)",5.333333,0.577350,50.313569,1.000000,0.870665,1.0,0.129335
2,Reddit,ift,hopfield_update,3,0.926777,0.003281,0.924985,0.005281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.000000,0.870665,1.0,0.129335
3,Reddit,ift,hnn,4,0.923994,0.005208,0.920888,0.006690,"(2, 2, 5)",3.000000,1.732051,69.798048,1.000000,0.870665,1.0,0.129335
4,Reddit,ift,lnn,5,0.919047,0.001441,0.915876,0.001166,"(6, 6, 6)",6.000000,0.000000,60.554952,1.000000,0.870665,1.0,0.129335
5,Reddit,settransformer,ift_update,6,0.895288,0.000231,0.897676,0.000370,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000,0.870665,1.0,0.129335
6,Reddit,sum,ift_update,7,0.889582,0.001279,0.891949,0.001674,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000,0.870665,1.0,0.129335
7,Reddit,ift,ift_update,8,0.883505,0.017277,0.867068,0.020027,"(5, 5, 6)",5.333333,0.577350,59.139892,1.000000,0.870665,1.0,0.129335
8,Reddit,hopfield,ift_update,9,0.870665,NaN,0.869175,NaN,"(2,)",2.000000,NaN,71.289740,0.333333,0.870665,1.0,0.129335



=== MOOC : IFT-involved models ranked by TEST MRR ===
test mrr range: min=0.940166, max=0.969200, width=0.029034


,dataset,aggregator,update,test_rank_among_ift,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,std_best_val_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,test_range_min,test_range_max,test_range_width
0,MOOC,ift,tgn_gru,1,0.969200,0.011324,0.973237,0.009487,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0,0.940166,0.9692,0.029034
1,MOOC,ift,hopfield_update,2,0.959932,0.007883,0.966239,0.005437,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0,0.940166,0.9692,0.029034
2,MOOC,ift,ift_update,3,0.958428,0.001274,0.964978,0.000469,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0,0.940166,0.9692,0.029034
3,MOOC,settransformer,ift_update,4,0.954992,0.001064,0.958664,0.001762,"(2, 4, 5)",3.666667,1.527525,78.565425,1.0,0.940166,0.9692,0.029034
4,MOOC,deepsets,ift_update,5,0.954384,0.001177,0.956429,0.003705,"(1, 1, 2)",1.333333,0.577350,42.312316,1.0,0.940166,0.9692,0.029034
5,MOOC,sum,ift_update,6,0.953874,0.001572,0.960063,0.001396,"(1, 1, 1)",1.000000,0.000000,40.129971,1.0,0.940166,0.9692,0.029034
6,MOOC,ift,hnn,7,0.949276,0.004420,0.954781,0.002781,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0,0.940166,0.9692,0.029034
7,MOOC,hopfield,ift_update,8,0.945867,0.001515,0.953094,0.000871,"(1, 2, 5)",2.666667,2.081666,53.740889,1.0,0.940166,0.9692,0.029034
8,MOOC,ift,lnn,9,0.940166,0.023555,0.953411,0.002691,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0,0.940166,0.9692,0.029034



=== LastFM : IFT-involved models ranked by TEST MRR ===
test mrr range: min=0.303972, max=1.000000, width=0.696028


,dataset,aggregator,update,test_rank_among_ift,mean_best_test_mrr,std_best_test_mrr,mean_best_val_mrr,std_best_val_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate,test_range_min,test_range_max,test_range_width
0,LastFM,deepsets,ift_update,1,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.0,0.303972,1.0,0.696028
1,LastFM,settransformer,ift_update,2,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.0,0.303972,1.0,0.696028
2,LastFM,ift,tgn_gru,3,0.475079,0.002489,0.440791,0.005729,"(2, 4, 6)",4.000000,2.000000,2513.622913,1.0,0.303972,1.0,0.696028
3,LastFM,ift,hopfield_update,4,0.465878,0.004632,0.433506,0.004482,"(2, 3, 3)",2.666667,0.577350,3204.043232,1.0,0.303972,1.0,0.696028
4,LastFM,ift,ift_update,5,0.462959,0.000596,0.425462,0.000290,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0,0.303972,1.0,0.696028
5,LastFM,ift,hnn,6,0.368250,0.052612,0.343373,0.044620,"(2, 4, 5)",3.666667,1.527525,3541.207378,1.0,0.303972,1.0,0.696028
6,LastFM,ift,lnn,7,0.350109,0.039379,0.330548,0.035811,"(5, 6, 6)",5.666667,0.577350,3022.797079,1.0,0.303972,1.0,0.696028
7,LastFM,sum,ift_update,8,0.303972,0.000744,0.268471,0.000587,"(4, 6, 6)",5.333333,1.154701,2468.323378,1.0,0.303972,1.0,0.696028


In [42]:
ift_combined_tables = {}

for ds in EXPECTED_DATASETS:
    ds_df = ift_models[ift_models["dataset"] == ds].copy()

    # ranks under both metrics
    ds_df["val_rank_among_ift"] = (
        ds_df["mean_best_val_mrr"].rank(ascending=False, method="min")
    )
    ds_df["test_rank_among_ift"] = (
        ds_df["mean_best_test_mrr"].rank(ascending=False, method="min")
    )

    ds_df = ds_df.sort_values(
        ["val_rank_among_ift", "test_rank_among_ift"],
        ascending=[True, True]
    ).reset_index(drop=True)

    out = ds_df[[
        "dataset",
        "aggregator",
        "update",
        "val_rank_among_ift",
        "test_rank_among_ift",
        "mean_best_val_mrr",
        "std_best_val_mrr",
        "mean_best_test_mrr",
        "std_best_test_mrr",
        "best_epoch_list",
        "mean_best_epoch",
        "std_best_epoch",
        "mean_wall_sec",
        "completion_rate",
    ]].copy()

    ift_combined_tables[ds] = out

    print(f"\n=== {ds} : IFT-involved combined ranking table ===")
    display(out)


=== Wikipedia : IFT-involved combined ranking table ===


,dataset,aggregator,update,val_rank_among_ift,test_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Wikipedia,ift,tgn_gru,1.0,1.0,0.860372,0.001454,0.866016,0.002074,"(2, 4, 6)",4.000000,2.000000,43.048286,1.0
1,Wikipedia,ift,hopfield_update,2.0,2.0,0.857577,0.003452,0.863980,0.002124,"(4, 5, 6)",5.000000,1.000000,397.996660,1.0
2,Wikipedia,ift,ift_update,3.0,3.0,0.815851,0.001271,0.828377,0.002290,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0
3,Wikipedia,ift,hnn,4.0,4.0,0.800154,0.006924,0.817659,0.010110,"(1, 1, 3)",1.666667,1.154701,57.269979,1.0
4,Wikipedia,ift,lnn,5.0,5.0,0.781483,0.002939,0.809615,0.002388,"(1, 2, 4)",2.333333,1.527525,49.873984,1.0
5,Wikipedia,sum,ift_update,6.0,6.0,0.721674,0.002442,0.740272,0.001980,"(1, 1, 1)",1.000000,0.000000,42.632306,1.0
6,Wikipedia,deepsets,ift_update,7.0,8.0,0.709918,0.001802,0.732147,0.001775,"(1, 4, 6)",3.666667,2.516611,44.296756,1.0
7,Wikipedia,settransformer,ift_update,8.0,7.0,0.709603,0.000825,0.738808,0.000770,"(1, 4, 4)",3.000000,1.732051,80.028924,1.0
8,Wikipedia,hopfield,ift_update,9.0,9.0,0.693130,0.003085,0.717308,0.003314,"(1, 1, 6)",2.666667,2.886751,53.113788,1.0



=== Reddit : IFT-involved combined ranking table ===


,dataset,aggregator,update,val_rank_among_ift,test_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000
1,Reddit,ift,tgn_gru,2.0,2.0,0.939561,0.003510,0.940129,0.003214,"(5, 5, 6)",5.333333,0.577350,50.313569,1.000000
2,Reddit,ift,hopfield_update,3.0,3.0,0.924985,0.005281,0.926777,0.003281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.000000
3,Reddit,ift,hnn,4.0,4.0,0.920888,0.006690,0.923994,0.005208,"(2, 2, 5)",3.000000,1.732051,69.798048,1.000000
4,Reddit,ift,lnn,5.0,5.0,0.915876,0.001166,0.919047,0.001441,"(6, 6, 6)",6.000000,0.000000,60.554952,1.000000
5,Reddit,settransformer,ift_update,6.0,6.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000
6,Reddit,sum,ift_update,7.0,7.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000
7,Reddit,hopfield,ift_update,8.0,9.0,0.869175,NaN,0.870665,NaN,"(2,)",2.000000,NaN,71.289740,0.333333
8,Reddit,ift,ift_update,9.0,8.0,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.577350,59.139892,1.000000



=== MOOC : IFT-involved combined ranking table ===


,dataset,aggregator,update,val_rank_among_ift,test_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,MOOC,ift,tgn_gru,1.0,1.0,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0
1,MOOC,ift,hopfield_update,2.0,2.0,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0
2,MOOC,ift,ift_update,3.0,3.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0
3,MOOC,sum,ift_update,4.0,6.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.0
4,MOOC,settransformer,ift_update,5.0,4.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.0
5,MOOC,deepsets,ift_update,6.0,5.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.0
6,MOOC,ift,hnn,7.0,7.0,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0
7,MOOC,ift,lnn,8.0,9.0,0.953411,0.002691,0.940166,0.023555,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0
8,MOOC,hopfield,ift_update,9.0,8.0,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.0



=== LastFM : IFT-involved combined ranking table ===


,dataset,aggregator,update,val_rank_among_ift,test_rank_among_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.0
1,LastFM,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.0
2,LastFM,ift,tgn_gru,3.0,3.0,0.440791,0.005729,0.475079,0.002489,"(2, 4, 6)",4.000000,2.000000,2513.622913,1.0
3,LastFM,ift,hopfield_update,4.0,4.0,0.433506,0.004482,0.465878,0.004632,"(2, 3, 3)",2.666667,0.577350,3204.043232,1.0
4,LastFM,ift,ift_update,5.0,5.0,0.425462,0.000290,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0
5,LastFM,ift,hnn,6.0,6.0,0.343373,0.044620,0.368250,0.052612,"(2, 4, 5)",3.666667,1.527525,3541.207378,1.0
6,LastFM,ift,lnn,7.0,7.0,0.330548,0.035811,0.350109,0.039379,"(5, 6, 6)",5.666667,0.577350,3022.797079,1.0
7,LastFM,sum,ift_update,8.0,8.0,0.268471,0.000587,0.303972,0.000744,"(4, 6, 6)",5.333333,1.154701,2468.323378,1.0


In [44]:
# -----------------------------------
# One combined table for all IFT-involved models
# "uses IFT in any capacity" means:
#   aggregator == "ift" OR update == "ift_update"
# -----------------------------------

ift_combined = leaderboard[
    (leaderboard["aggregator"] == "ift") | (leaderboard["update"] == "ift_update")
].copy()

# add per-dataset ranks among only the IFT-involved rows
ift_combined["val_rank_within_dataset_ift"] = (
    ift_combined.groupby("dataset")["mean_best_val_mrr"]
    .rank(ascending=False, method="min")
)

ift_combined["test_rank_within_dataset_ift"] = (
    ift_combined.groupby("dataset")["mean_best_test_mrr"]
    .rank(ascending=False, method="min")
)

# optional: label HOW it uses IFT
def classify_ift_usage(row):
    agg_ift = row["aggregator"] == "ift"
    upd_ift = row["update"] == "ift_update"
    if agg_ift and upd_ift:
        return "ift_both"
    elif agg_ift:
        return "ift_agg_only"
    elif upd_ift:
        return "ift_update_only"
    return "not_ift"

ift_combined["ift_usage"] = ift_combined.apply(classify_ift_usage, axis=1)

# nice readable combined table
ift_combined = ift_combined[[
    "dataset",
    "ift_usage",
    "aggregator",
    "update",
    "val_rank_within_dataset_ift",
    "test_rank_within_dataset_ift",
    "mean_best_val_mrr",
    "std_best_val_mrr",
    "mean_best_test_mrr",
    "std_best_test_mrr",
    "best_epoch_list",
    "mean_best_epoch",
    "std_best_epoch",
    "mean_wall_sec",
    "completion_rate",
]].sort_values(
    ["dataset", "val_rank_within_dataset_ift", "test_rank_within_dataset_ift"],
    ascending=[True, True, True]
).reset_index(drop=True)

display(
    ift_combined.sort_values(
        ["mean_best_val_mrr", "mean_best_test_mrr"],
        ascending=False
    ).reset_index(drop=True)
)

,dataset,ift_usage,aggregator,update,val_rank_within_dataset_ift,test_rank_within_dataset_ift,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift_update_only,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.000000
1,LastFM,ift_update_only,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.000000
2,Reddit,ift_update_only,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000
3,MOOC,ift_agg_only,ift,tgn_gru,1.0,1.0,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.000000
4,MOOC,ift_agg_only,ift,hopfield_update,2.0,2.0,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.000000
5,MOOC,ift_both,ift,ift_update,3.0,3.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.000000
6,MOOC,ift_update_only,sum,ift_update,4.0,6.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.000000
7,MOOC,ift_update_only,settransformer,ift_update,5.0,4.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.000000
8,MOOC,ift_update_only,deepsets,ift_update,6.0,5.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.000000
9,MOOC,ift_agg_only,ift,hnn,7.0,7.0,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.000000


In [45]:
ift_usage_summary = (
    ift_combined.groupby(["ift_usage", "aggregator", "update"], dropna=False)
    .agg(
        datasets_seen=("dataset", "nunique"),
        mean_val_mrr=("mean_best_val_mrr", "mean"),
        std_val_mrr_across_datasets=("mean_best_val_mrr", "std"),
        mean_test_mrr=("mean_best_test_mrr", "mean"),
        mean_best_epoch=("mean_best_epoch", "mean"),
        mean_wall_sec=("mean_wall_sec", "mean"),
        mean_completion_rate=("completion_rate", "mean"),
    )
    .reset_index()
    .sort_values(["mean_val_mrr", "mean_test_mrr"], ascending=False)
    .reset_index(drop=True)
)

display(ift_usage_summary)

,ift_usage,aggregator,update,datasets_seen,mean_val_mrr,std_val_mrr_across_datasets,mean_test_mrr,mean_best_epoch,mean_wall_sec,mean_completion_rate
0,ift_update_only,deepsets,ift_update,4,0.916587,0.139302,0.921633,2.000000,658.397227,1.000000
1,ift_update_only,settransformer,ift_update,4,0.891486,0.128333,0.897272,3.250000,1087.526922,1.000000
2,ift_update_only,hopfield,ift_update,3,0.838466,0.132675,0.844613,2.444444,59.381473,0.777778
3,ift_agg_only,ift,tgn_gru,4,0.803491,0.246384,0.812606,4.416667,662.134598,1.000000
4,ift_agg_only,ift,hopfield_update,4,0.795577,0.245501,0.804142,3.750000,1098.050700,1.000000
5,ift_both,ift,ift_update,4,0.768340,0.236810,0.783317,3.583333,724.231590,1.000000
6,ift_agg_only,ift,hnn,4,0.754799,0.282198,0.764795,2.750000,931.079048,1.000000
7,ift_agg_only,ift,lnn,4,0.745330,0.286202,0.754734,4.416667,795.466427,1.000000
8,ift_update_only,sum,ift_update,4,0.710539,0.311298,0.721925,3.166667,650.429441,1.000000


In [47]:
# -----------------------------------
# Split IFT-involved models into 3 tables:
# 1) IFT as update only
# 2) IFT as aggregator only
# 3) IFT as both
# -----------------------------------

ift_update_only = leaderboard[
    (leaderboard["update"] == "ift_update") &
    (leaderboard["aggregator"] != "ift")
].copy()

ift_agg_only = leaderboard[
    (leaderboard["aggregator"] == "ift") &
    (leaderboard["update"] != "ift_update")
].copy()

ift_both = leaderboard[
    (leaderboard["aggregator"] == "ift") &
    (leaderboard["update"] == "ift_update")
].copy()


def add_within_dataset_ranks(df):
    df = df.copy()

    if len(df) == 0:
        return df

    df["val_rank_within_group_dataset"] = (
        df.groupby("dataset")["mean_best_val_mrr"]
        .rank(ascending=False, method="min")
    )

    df["test_rank_within_group_dataset"] = (
        df.groupby("dataset")["mean_best_test_mrr"]
        .rank(ascending=False, method="min")
    )

    return df


ift_update_only = add_within_dataset_ranks(ift_update_only)
ift_agg_only = add_within_dataset_ranks(ift_agg_only)
ift_both = add_within_dataset_ranks(ift_both)


common_cols = [
    "dataset",
    "aggregator",
    "update",
    "val_rank_within_group_dataset",
    "test_rank_within_group_dataset",
    "mean_best_val_mrr",
    "std_best_val_mrr",
    "mean_best_test_mrr",
    "std_best_test_mrr",
    "best_epoch_list",
    "mean_best_epoch",
    "std_best_epoch",
    "mean_wall_sec",
    "completion_rate",
]

ift_update_only = (
    ift_update_only[common_cols]
    .sort_values(["dataset", "val_rank_within_group_dataset", "test_rank_within_group_dataset"])
    .reset_index(drop=True)
)

ift_agg_only = (
    ift_agg_only[common_cols]
    .sort_values(["dataset", "val_rank_within_group_dataset", "test_rank_within_group_dataset"])
    .reset_index(drop=True)
)

ift_both = (
    ift_both[common_cols]
    .sort_values(["dataset", "val_rank_within_group_dataset", "test_rank_within_group_dataset"])
    .reset_index(drop=True)
)

print("=== IFT as UPDATE only ===")
display(ift_update_only)

print("=== IFT as AGGREGATOR only ===")
display(ift_agg_only)

print("=== IFT as BOTH aggregator and update ===")
display(ift_both)

=== IFT as UPDATE only ===


,dataset,aggregator,update,val_rank_within_group_dataset,test_rank_within_group_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.000000
1,LastFM,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.000000
2,LastFM,sum,ift_update,3.0,3.0,0.268471,0.000587,0.303972,0.000744,"(4, 6, 6)",5.333333,1.154701,2468.323378,1.000000
3,MOOC,sum,ift_update,1.0,3.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.000000
4,MOOC,settransformer,ift_update,2.0,1.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.000000
5,MOOC,deepsets,ift_update,3.0,2.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.000000
6,MOOC,hopfield,ift_update,4.0,4.0,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.000000
7,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000
8,Reddit,settransformer,ift_update,2.0,2.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000
9,Reddit,sum,ift_update,3.0,3.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000


=== IFT as AGGREGATOR only ===


,dataset,aggregator,update,val_rank_within_group_dataset,test_rank_within_group_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,tgn_gru,1.0,1.0,0.440791,0.005729,0.475079,0.002489,"(2, 4, 6)",4.000000,2.000000,2513.622913,1.0
1,LastFM,ift,hopfield_update,2.0,2.0,0.433506,0.004482,0.465878,0.004632,"(2, 3, 3)",2.666667,0.577350,3204.043232,1.0
2,LastFM,ift,hnn,3.0,3.0,0.343373,0.044620,0.368250,0.052612,"(2, 4, 5)",3.666667,1.527525,3541.207378,1.0
3,LastFM,ift,lnn,4.0,4.0,0.330548,0.035811,0.350109,0.039379,"(5, 6, 6)",5.666667,0.577350,3022.797079,1.0
4,MOOC,ift,tgn_gru,1.0,1.0,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0
5,MOOC,ift,hopfield_update,2.0,2.0,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0
6,MOOC,ift,hnn,3.0,3.0,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0
7,MOOC,ift,lnn,4.0,4.0,0.953411,0.002691,0.940166,0.023555,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0
8,Reddit,ift,tgn_gru,1.0,1.0,0.939561,0.003510,0.940129,0.003214,"(5, 5, 6)",5.333333,0.577350,50.313569,1.0
9,Reddit,ift,hopfield_update,2.0,2.0,0.924985,0.005281,0.926777,0.003281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.0


=== IFT as BOTH aggregator and update ===


,dataset,aggregator,update,val_rank_within_group_dataset,test_rank_within_group_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,ift_update,1.0,1.0,0.425462,0.000290,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0
1,MOOC,ift,ift_update,1.0,1.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0
2,Reddit,ift,ift_update,1.0,1.0,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.577350,59.139892,1.0
3,Wikipedia,ift,ift_update,1.0,1.0,0.815851,0.001271,0.828377,0.002290,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0


In [48]:
display(ift_update_only.sort_values("mean_best_val_mrr", ascending=False).reset_index(drop=True))
display(ift_agg_only.sort_values("mean_best_val_mrr", ascending=False).reset_index(drop=True))
display(ift_both.sort_values("mean_best_val_mrr", ascending=False).reset_index(drop=True))

,dataset,aggregator,update,val_rank_within_group_dataset,test_rank_within_group_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.000000
1,LastFM,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.000000
2,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000
3,MOOC,sum,ift_update,1.0,3.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.000000
4,MOOC,settransformer,ift_update,2.0,1.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.000000
5,MOOC,deepsets,ift_update,3.0,2.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.000000
6,MOOC,hopfield,ift_update,4.0,4.0,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.000000
7,Reddit,settransformer,ift_update,2.0,2.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000
8,Reddit,sum,ift_update,3.0,3.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000
9,Reddit,hopfield,ift_update,4.0,4.0,0.869175,NaN,0.870665,NaN,"(2,)",2.000000,NaN,71.289740,0.333333


,dataset,aggregator,update,val_rank_within_group_dataset,test_rank_within_group_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,MOOC,ift,tgn_gru,1.0,1.0,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0
1,MOOC,ift,hopfield_update,2.0,2.0,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0
2,MOOC,ift,hnn,3.0,3.0,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0
3,MOOC,ift,lnn,4.0,4.0,0.953411,0.002691,0.940166,0.023555,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0
4,Reddit,ift,tgn_gru,1.0,1.0,0.939561,0.003510,0.940129,0.003214,"(5, 5, 6)",5.333333,0.577350,50.313569,1.0
5,Reddit,ift,hopfield_update,2.0,2.0,0.924985,0.005281,0.926777,0.003281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.0
6,Reddit,ift,hnn,3.0,3.0,0.920888,0.006690,0.923994,0.005208,"(2, 2, 5)",3.000000,1.732051,69.798048,1.0
7,Reddit,ift,lnn,4.0,4.0,0.915876,0.001166,0.919047,0.001441,"(6, 6, 6)",6.000000,0.000000,60.554952,1.0
8,Wikipedia,ift,tgn_gru,1.0,1.0,0.860372,0.001454,0.866016,0.002074,"(2, 4, 6)",4.000000,2.000000,43.048286,1.0
9,Wikipedia,ift,hopfield_update,2.0,2.0,0.857577,0.003452,0.863980,0.002124,"(4, 5, 6)",5.000000,1.000000,397.996660,1.0


,dataset,aggregator,update,val_rank_within_group_dataset,test_rank_within_group_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,MOOC,ift,ift_update,1.0,1.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0
1,Reddit,ift,ift_update,1.0,1.0,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.577350,59.139892,1.0
2,Wikipedia,ift,ift_update,1.0,1.0,0.815851,0.001271,0.828377,0.002290,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0
3,LastFM,ift,ift_update,1.0,1.0,0.425462,0.000290,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0


In [49]:
# -----------------------------------
# Add ranks within each dataset across ALL model combos
# -----------------------------------

leaderboard["val_rank_all_models_in_dataset"] = (
    leaderboard.groupby("dataset")["mean_best_val_mrr"]
    .rank(ascending=False, method="min")
)

leaderboard["test_rank_all_models_in_dataset"] = (
    leaderboard.groupby("dataset")["mean_best_test_mrr"]
    .rank(ascending=False, method="min")
)

display(
    leaderboard[[
        "dataset", "aggregator", "update",
        "mean_best_val_mrr", "mean_best_test_mrr",
        "val_rank_all_models_in_dataset",
        "test_rank_all_models_in_dataset"
    ]].head(10)
)

,dataset,aggregator,update,mean_best_val_mrr,mean_best_test_mrr,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset
0,LastFM,deepsets,hnn,0.236944,0.251735,21.0,22.0
1,LastFM,deepsets,hopfield_update,1.000000,1.000000,1.0,1.0
2,LastFM,deepsets,ift_update,1.000000,1.000000,1.0,1.0
3,LastFM,deepsets,lnn,0.230624,0.259484,23.0,21.0
4,LastFM,deepsets,tgn_gru,0.389457,0.412289,11.0,11.0
5,LastFM,hopfield,hnn,0.300555,0.337531,16.0,16.0
6,LastFM,hopfield,hopfield_update,0.413990,0.443000,9.0,9.0
7,LastFM,hopfield,lnn,0.248813,0.267674,20.0,20.0
8,LastFM,hopfield,tgn_gru,0.397860,0.427766,10.0,10.0
9,LastFM,ift,hnn,0.343373,0.368250,14.0,14.0


In [50]:
# -----------------------------------
# 3 IFT tables using ranks among ALL 25 models in each dataset
# -----------------------------------

ift_update_only = leaderboard[
    (leaderboard["update"] == "ift_update") &
    (leaderboard["aggregator"] != "ift")
].copy()

ift_agg_only = leaderboard[
    (leaderboard["aggregator"] == "ift") &
    (leaderboard["update"] != "ift_update")
].copy()

ift_both = leaderboard[
    (leaderboard["aggregator"] == "ift") &
    (leaderboard["update"] == "ift_update")
].copy()

common_cols = [
    "dataset",
    "aggregator",
    "update",
    "val_rank_all_models_in_dataset",
    "test_rank_all_models_in_dataset",
    "mean_best_val_mrr",
    "std_best_val_mrr",
    "mean_best_test_mrr",
    "std_best_test_mrr",
    "best_epoch_list",
    "mean_best_epoch",
    "std_best_epoch",
    "mean_wall_sec",
    "completion_rate",
]

ift_update_only = (
    ift_update_only[common_cols]
    .sort_values(
        ["dataset", "val_rank_all_models_in_dataset", "test_rank_all_models_in_dataset"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

ift_agg_only = (
    ift_agg_only[common_cols]
    .sort_values(
        ["dataset", "val_rank_all_models_in_dataset", "test_rank_all_models_in_dataset"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

ift_both = (
    ift_both[common_cols]
    .sort_values(
        ["dataset", "val_rank_all_models_in_dataset", "test_rank_all_models_in_dataset"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

print("=== IFT as UPDATE only ===")
display(ift_update_only)

print("=== IFT as AGGREGATOR only ===")
display(ift_agg_only)

print("=== IFT as BOTH ===")
display(ift_both)

=== IFT as UPDATE only ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.000000
1,LastFM,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.000000
2,LastFM,sum,ift_update,19.0,18.0,0.268471,0.000587,0.303972,0.000744,"(4, 6, 6)",5.333333,1.154701,2468.323378,1.000000
3,MOOC,sum,ift_update,12.0,14.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.000000
4,MOOC,settransformer,ift_update,13.0,11.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.000000
5,MOOC,deepsets,ift_update,14.0,12.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.000000
6,MOOC,hopfield,ift_update,20.0,19.0,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.000000
7,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000
8,Reddit,settransformer,ift_update,11.0,12.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000
9,Reddit,sum,ift_update,13.0,13.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000


=== IFT as AGGREGATOR only ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,tgn_gru,4.0,4.0,0.440791,0.005729,0.475079,0.002489,"(2, 4, 6)",4.000000,2.000000,2513.622913,1.0
1,LastFM,ift,hopfield_update,6.0,5.0,0.433506,0.004482,0.465878,0.004632,"(2, 3, 3)",2.666667,0.577350,3204.043232,1.0
2,LastFM,ift,hnn,14.0,14.0,0.343373,0.044620,0.368250,0.052612,"(2, 4, 5)",3.666667,1.527525,3541.207378,1.0
3,LastFM,ift,lnn,15.0,15.0,0.330548,0.035811,0.350109,0.039379,"(5, 6, 6)",5.666667,0.577350,3022.797079,1.0
4,MOOC,ift,tgn_gru,1.0,1.0,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0
5,MOOC,ift,hopfield_update,4.0,4.0,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0
6,MOOC,ift,hnn,17.0,16.0,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0
7,MOOC,ift,lnn,19.0,21.0,0.953411,0.002691,0.940166,0.023555,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0
8,Reddit,ift,tgn_gru,2.0,2.0,0.939561,0.003510,0.940129,0.003214,"(5, 5, 6)",5.333333,0.577350,50.313569,1.0
9,Reddit,ift,hopfield_update,3.0,3.0,0.924985,0.005281,0.926777,0.003281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.0


=== IFT as BOTH ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,ift_update,8.0,6.0,0.425462,0.000290,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0
1,MOOC,ift,ift_update,7.0,6.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0
2,Reddit,ift,ift_update,25.0,17.0,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.577350,59.139892,1.0
3,Wikipedia,ift,ift_update,11.0,8.0,0.815851,0.001271,0.828377,0.002290,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0


In [51]:
for ds in EXPECTED_DATASETS:
    print(f"\n=== IFT update only : {ds} ===")
    display(
        ift_update_only[ift_update_only["dataset"] == ds]
        .reset_index(drop=True)
    )


=== IFT update only : Wikipedia ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Wikipedia,sum,ift_update,14.0,14.0,0.721674,0.002442,0.740272,0.001980,"(1, 1, 1)",1.000000,0.000000,42.632306,1.0
1,Wikipedia,deepsets,ift_update,16.0,17.0,0.709918,0.001802,0.732147,0.001775,"(1, 4, 6)",3.666667,2.516611,44.296756,1.0
2,Wikipedia,settransformer,ift_update,17.0,15.0,0.709603,0.000825,0.738808,0.000770,"(1, 4, 4)",3.000000,1.732051,80.028924,1.0
3,Wikipedia,hopfield,ift_update,18.0,18.0,0.693130,0.003085,0.717308,0.003314,"(1, 1, 6)",2.666667,2.886751,53.113788,1.0



=== IFT update only : Reddit ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.00000,54.240344,1.000000
1,Reddit,settransformer,ift_update,11.0,12.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.57735,116.470368,1.000000
2,Reddit,sum,ift_update,13.0,13.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.57735,50.632109,1.000000
3,Reddit,hopfield,ift_update,23.0,25.0,0.869175,NaN,0.870665,NaN,"(2,)",2.000000,NaN,71.289740,0.333333



=== IFT update only : MOOC ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,MOOC,sum,ift_update,12.0,14.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.0
1,MOOC,settransformer,ift_update,13.0,11.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.0
2,MOOC,deepsets,ift_update,14.0,12.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.0
3,MOOC,hopfield,ift_update,20.0,19.0,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.0



=== IFT update only : LastFM ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.0
1,LastFM,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.0
2,LastFM,sum,ift_update,19.0,18.0,0.268471,0.000587,0.303972,0.000744,"(4, 6, 6)",5.333333,1.154701,2468.323378,1.0


In [56]:
# IFT update only — sorted by VAL
display(
    ift_update_only.sort_values(
        ["mean_best_val_mrr", "mean_best_test_mrr"],
        ascending=[False, False]
    ).reset_index(drop=True)
)

# IFT agg only — sorted by VAL
display(
    ift_agg_only.sort_values(
        ["dataset", "mean_best_val_mrr", "mean_best_test_mrr"],
        ascending=[True, False, False]
    ).reset_index(drop=True)
)

# IFT both — sorted by VAL
display(
    ift_both.sort_values(
        ["dataset", "mean_best_val_mrr", "mean_best_test_mrr"],
        ascending=[True, False, False]
    ).reset_index(drop=True)
)

,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,2492.739492,1.000000
1,LastFM,settransformer,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(1, 1, 1)",1.000000,0.000000,4075.042972,1.000000
2,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.000000,54.240344,1.000000
3,MOOC,sum,ift_update,12.0,14.0,0.960063,0.001396,0.953874,0.001572,"(1, 1, 1)",1.000000,0.000000,40.129971,1.000000
4,MOOC,settransformer,ift_update,13.0,11.0,0.958664,0.001762,0.954992,0.001064,"(2, 4, 5)",3.666667,1.527525,78.565425,1.000000
5,MOOC,deepsets,ift_update,14.0,12.0,0.956429,0.003705,0.954384,0.001177,"(1, 1, 2)",1.333333,0.577350,42.312316,1.000000
6,MOOC,hopfield,ift_update,20.0,19.0,0.953094,0.000871,0.945867,0.001515,"(1, 2, 5)",2.666667,2.081666,53.740889,1.000000
7,Reddit,settransformer,ift_update,11.0,12.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.577350,116.470368,1.000000
8,Reddit,sum,ift_update,13.0,13.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.577350,50.632109,1.000000
9,Reddit,hopfield,ift_update,23.0,25.0,0.869175,NaN,0.870665,NaN,"(2,)",2.000000,NaN,71.289740,0.333333


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,tgn_gru,4.0,4.0,0.440791,0.005729,0.475079,0.002489,"(2, 4, 6)",4.000000,2.000000,2513.622913,1.0
1,LastFM,ift,hopfield_update,6.0,5.0,0.433506,0.004482,0.465878,0.004632,"(2, 3, 3)",2.666667,0.577350,3204.043232,1.0
2,LastFM,ift,hnn,14.0,14.0,0.343373,0.044620,0.368250,0.052612,"(2, 4, 5)",3.666667,1.527525,3541.207378,1.0
3,LastFM,ift,lnn,15.0,15.0,0.330548,0.035811,0.350109,0.039379,"(5, 6, 6)",5.666667,0.577350,3022.797079,1.0
4,MOOC,ift,tgn_gru,1.0,1.0,0.973237,0.009487,0.969200,0.011324,"(2, 5, 6)",4.333333,2.081666,41.553625,1.0
5,MOOC,ift,hopfield_update,4.0,4.0,0.966239,0.005437,0.959932,0.007883,"(1, 2, 6)",3.000000,2.645751,245.753235,1.0
6,MOOC,ift,hnn,17.0,16.0,0.954781,0.002781,0.949276,0.004420,"(2, 2, 4)",2.666667,1.154701,56.040788,1.0
7,MOOC,ift,lnn,19.0,21.0,0.953411,0.002691,0.940166,0.023555,"(2, 4, 5)",3.666667,1.527525,48.639695,1.0
8,Reddit,ift,tgn_gru,2.0,2.0,0.939561,0.003510,0.940129,0.003214,"(5, 5, 6)",5.333333,0.577350,50.313569,1.0
9,Reddit,ift,hopfield_update,3.0,3.0,0.924985,0.005281,0.926777,0.003281,"(3, 5, 5)",4.333333,1.154701,544.409674,1.0


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,ift_update,8.0,6.0,0.425462,0.000290,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0
1,MOOC,ift,ift_update,7.0,6.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0
2,Reddit,ift,ift_update,25.0,17.0,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.577350,59.139892,1.0
3,Wikipedia,ift,ift_update,11.0,8.0,0.815851,0.001271,0.828377,0.002290,"(1, 1, 2)",1.333333,0.577350,47.539144,1.0


In [59]:
for ds in EXPECTED_DATASETS:
    print(f"\n=== IFT update only : {ds} (sorted by val) ===")
    display(
        ift_update_only[ift_update_only["dataset"] == ds]
        .sort_values(
            ["mean_best_val_mrr", "mean_best_test_mrr"],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )

for ds in EXPECTED_DATASETS:
    print(f"\n=== IFT aggre only : {ds} (sorted by val) ===")
    display(
        ift_agg_only[ift_agg_only["dataset"] == ds]
        .sort_values(
            ["mean_best_val_mrr", "mean_best_test_mrr"],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )

for ds in EXPECTED_DATASETS:
    print(f"\n=== Both IFT : {ds} (sorted by val) ===")
    display(
        ift_both[ift_both["dataset"] == ds]
        .sort_values(
            ["mean_best_val_mrr", "mean_best_test_mrr"],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )


=== IFT update only : Wikipedia (sorted by val) ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Wikipedia,sum,ift_update,14.0,14.0,0.721674,0.002442,0.740272,0.001980,"(1, 1, 1)",1.000000,0.000000,42.632306,1.0
1,Wikipedia,deepsets,ift_update,16.0,17.0,0.709918,0.001802,0.732147,0.001775,"(1, 4, 6)",3.666667,2.516611,44.296756,1.0
2,Wikipedia,settransformer,ift_update,17.0,15.0,0.709603,0.000825,0.738808,0.000770,"(1, 4, 4)",3.000000,1.732051,80.028924,1.0
3,Wikipedia,hopfield,ift_update,18.0,18.0,0.693130,0.003085,0.717308,0.003314,"(1, 1, 6)",2.666667,2.886751,53.113788,1.0



=== IFT update only : Reddit (sorted by val) ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Reddit,deepsets,ift_update,1.0,1.0,1.000000,0.000000,1.000000,0.000000,"(2, 2, 2)",2.000000,0.00000,54.240344,1.000000
1,Reddit,settransformer,ift_update,11.0,12.0,0.897676,0.000370,0.895288,0.000231,"(5, 5, 6)",5.333333,0.57735,116.470368,1.000000
2,Reddit,sum,ift_update,13.0,13.0,0.891949,0.001674,0.889582,0.001279,"(5, 5, 6)",5.333333,0.57735,50.632109,1.000000
3,Reddit,hopfield,ift_update,23.0,25.0,0.869175,NaN,0.870665,NaN,"(2,)",2.000000,NaN,71.289740,0.333333



=== IFT update only : MOOC (sorted by val) ===


...

,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,Reddit,ift,ift_update,25.0,17.0,0.867068,0.020027,0.883505,0.017277,"(5, 5, 6)",5.333333,0.57735,59.139892,1.0



=== Both IFT : MOOC (sorted by val) ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,MOOC,ift,ift_update,7.0,6.0,0.964978,0.000469,0.958428,0.001274,"(2, 4, 4)",3.333333,1.154701,45.744883,1.0



=== Both IFT : LastFM (sorted by val) ===


,dataset,aggregator,update,val_rank_all_models_in_dataset,test_rank_all_models_in_dataset,mean_best_val_mrr,std_best_val_mrr,mean_best_test_mrr,std_best_test_mrr,best_epoch_list,mean_best_epoch,std_best_epoch,mean_wall_sec,completion_rate
0,LastFM,ift,ift_update,8.0,6.0,0.425462,0.00029,0.462959,0.000596,"(3, 5, 5)",4.333333,1.154701,2744.502441,1.0
